In [ ]:
DOMAIN = "https://hamperedly-nonuniform-hyo.ngrok-free.dev"

### Install environment

In [ ]:
import importlib.util
import subprocess
import sys


def run_pip(*args):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "-qq", *args],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )


def run_cmd(cmd, shell: bool = False):
    subprocess.check_call(cmd, shell=shell)


if importlib.util.find_spec("flashrank") is None:
    run_pip("install", "vllm==0.10.0")
    run_pip("install", "triton==3.2.0")
    run_pip(
        "install",
        "flashrank",
        "langchain",
        "langchain-community",
        "langchain_google_genai",
        "openai",
        "faiss-cpu",
        "langchain_huggingface",
        "crawl4ai",
        "unidecode",
        "pymupdf4llm",
        "google-genai",
        "rapidfuzz",
        "transformers==4.57.0",
    )
    run_pip("uninstall", "-y", "openai")
    run_pip("install", "openai==1.90.0")
    run_cmd([
        "wget",
        "-q",
        "-O",
        "ngrok.zip",
        "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip",
    ])
    run_cmd(["unzip", "-o", "ngrok.zip"])
    run_cmd(["mv", "ngrok", "/usr/local/bin/ngrok"])
    run_cmd(["chmod", "+x", "/usr/local/bin/ngrok"])
    run_cmd("ps aux | grep ngrok", shell=True)
else:
    print("All libs aldready installed")
import os
# Fixed, do not change
os.environ["GLOO_SOCKET_NAME"] = "eth0"
os.environ["NCCL_SOCKET_NAME"] = "eth0"
os.environ["VLLM_HOST_IP"] = "127.0.0.1" # Internal ip for data communicate between VLLM components
os.environ["VLLM_USE_V1"] = "0" # T4 have compute capacity of 7.5, it need at least 8.0 to use V1

All libs aldready installed


##### Download package from server

In [3]:
IS_LOCAL = DOMAIN == "http://127.0.0.1:8000"
BASE_PATH = "" if IS_LOCAL else "/kaggle/working/"

In [ ]:
import requests
import io
import tarfile
import shutil
def unpack_folder(data: bytes, path: str):
    if os.path.exists(path): # Remove old code
        shutil.rmtree(path)
    with io.BytesIO(data) as tar_buffer:
        with tarfile.open(fileobj=tar_buffer, mode='r:gz') as tar:
            tar.extractall(path=path)
def unpack_file(data: bytes, path: str):
    os.makedirs(f"{BASE_PATH}files", exist_ok=True)
    if os.path.exists(path):
        os.remove(path)
    with open(f"{BASE_PATH}files/{path}", 'wb') as file:
        file.write(data)
def unpack_list(*names: str):
    # if DOMAIN == "http://127.0.0.1:8000": return
    for name in names:
        if "." in name:
            url = f"{DOMAIN}/package/{name}"
        else:
            url = f"{DOMAIN}/package/{name}"
        data = requests.get(url).content
        if "." in name:
            unpack_file(data, name)
        else:
            unpack_folder(data, name)
if not os.path.exists(f"{BASE_PATH}/files"):
    """"""
    # This consume a lot of quota. So only download when necessary
    unpack_list(
        "worker.env", "school_name.json", "school_alias.json", "local.pkl",
        "data_retriever", "server", "instruction", "school_mapper", 
        "lora/reader_v3", "lora/qwen_reranker_06b_v1"
    )

In [5]:
unpack_list("data_retriever","instruction","worker.env")

In [6]:
!pip install -U "landingai-ade"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 3.1 MB/s eta 0:00:00


In [7]:
# !pip install -U numpy==1.26.4 scipy==1.13.1 scikit-learn==1.5.1 \
# transformers==4.53.3 tokenizers==0.21.1 \
# --force-reinstall --no-cache-dir
# !pip install rapidfuzz
# !pip install pdfplumber

### Config

Load environment variable (api keys)

In [8]:
from dotenv import load_dotenv
load_dotenv(f"{BASE_PATH}files/worker.env")

True

Setup ngrok

In [9]:
NGROK_PORT = 8002
if DOMAIN != "http://127.0.0.1:8000":
    import subprocess
    subprocess.run(["ngrok", "config", "add-authtoken", os.getenv("NGROK_TOKEN_1", "")])
    subprocess.Popen(["ngrok", "http", str(NGROK_PORT)], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


Login hugging face

In [10]:
cmd = [
    "hf", "auth", "login",
    "--token", os.getenv("HUGGING_FACE_TOKEN")
]
import subprocess
subprocess.run(cmd)
print("")


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `Nam Hai Nguyen` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Nam Hai Nguyen`


##### Config

In [ ]:
from data_retriever import *
from server import *
from school_mapper import SchoolMapper
from typing import AsyncGenerator, NotRequired, Protocol
from typing import Callable, AsyncGenerator
from openai import AsyncOpenAI, OpenAI
from google import genai
from google.genai import types
import os
import pickle
import json
import asyncio
import enum
import traceback
import copy
import torch
import gc
from sentence_transformers import CrossEncoder
from vllm.lora.request import LoRARequest

2025-12-02 16:07:50.982586: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764691671.004257    1638 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764691671.010851    1638 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Consider using the pymupdf_layout package for a greatly improved page layout analysis.


In [12]:
from typing import Protocol, AsyncGenerator, TypedDict
class KeywordInfo(TypedDict):
    query: str
    priority: float
    info: str
    school: str
class KeywordModelProtocol(Protocol):
    async def keywords(self, question: str, params: GenerationParams, threshold: float = 0.5) -> list[KeywordInfo]: ...
class RouterModelProtocol(Protocol):
    async def route(self, question: str, params: GenerationParams) -> list[dict]: ...

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B"
# Retriever config
search_config = WebsearchConfig(
    page_timeout=15,
    file_timeout=15,
)
rag_config = RagConfig(
    embedding_name="intfloat/multilingual-e5-small",
    device="cuda",  # move embedding to CPU to avoid GPU OOM (CUBLAS alloc failed)
    use_hybrid=True,
    hybrid_alpha=0.7
)
splitter_config = SplitterConfig(
    tokenizer_name=MODEL_ID,
    chunk_size=512,
    chunk_overlap=64,
)
table_merge_config = MergeTableConfig(
    k_max_previous=5,
    k_max_next=5
)
neighbor_config = MergeNeighborConfig(
    k_previous_chunks=1,
    k_next_chunks=1
)
# Sampling Params
PAGE_RERANKER_PARAMS = {
    "temperature": 0.7,
    "top_p": 0.9,
    "max_tokens": 4096
}
KEYWORDS_PARAMS = {
    "temperature": 0.5,
    "top_p": 0.9,
    "max_tokens": 4096
}
ROUTER_PARAMS = {
    "temperature": 0.7,
    "top_p": 0.9,
    "max_tokens": 1024
}
SEP = "$$$"
MODELS: list[ModelInfo] = [
    {
        "name": "Qwen3 4B",
        "id": "Qwen/Qwen3-4B"
    },
    {
        "name": "Qwen3 4B LoRA",
        "id": f"Qwen/Qwen3-4B{SEP}1"
    }
]
CLIENT_INFO: WorkerServerInfo = {
    "name": "Test Qwen4B",
    "domain": "http://127.0.0.1:8002", # Auto change when run with ngrok
    "models": MODELS
}
READER_LORA = LoRARequest(
    lora_int_id= 1,
    lora_name= "Qwen Reader Lora",
    lora_path= f"{BASE_PATH}lora/reader_v3"
)
LORA_MAP = {
    1: READER_LORA
}

##### Template, instruction, prefix

In [14]:
from instruction import *

### Utility class

##### Data Retriever

In [ ]:
class LocalRetriever:
    """Search in static db using RAG with semantic search"""
    def __init__(self, rag_config: RagConfig = None) -> None:
        # Load documents
        with open(f"{BASE_PATH}files/local.pkl", 'rb') as file:
            self.all_docs = pickle.load(file)
        
        print(f"[LocalRetriever] Loaded {len(self.all_docs)} documents from local.pkl")
        
        # Initialize RAG components
        if rag_config is None:
            rag_config = RagConfig(
                embedding_name="intfloat/multilingual-e5-small",
                device="cuda",
                use_hybrid=True,
                hybrid_alpha=0.7
            )
        self.rag_config = rag_config
        
        # Initialize embedding model
        from langchain_huggingface import HuggingFaceEmbeddings
        self.embedding = HuggingFaceEmbeddings(
            model_name=rag_config.embedding_name,
            model_kwargs={"device": rag_config.device}
        )
        
        # Build FAISS vector store from all documents
        print("[LocalRetriever] Building FAISS vector store...")
        from langchain_community.vectorstores import FAISS
        self.vectorstore = FAISS.from_documents(self.all_docs, self.embedding)
        print(f"[LocalRetriever] Vector store built with {len(self.all_docs)} documents")
        
        # Initialize retriever with configurable k
        self.retriever = self.vectorstore.as_retriever(
            search_kwargs={"k": 10}  # Default retrieve top 10
        )
    
    def _filter_docs(self, school_id: str = None, section: str = None) -> list:
        """Filter documents by school_id and/or section"""
        filtered_docs = self.all_docs
        
        if school_id:
            filtered_docs = [doc for doc in filtered_docs 
                           if doc.metadata.get("school_id") == school_id]
        
        if section:
            filtered_docs = [doc for doc in filtered_docs 
                           if doc.metadata.get("section") == section]
        
        return filtered_docs
    
    def _semantic_search(self, query: str, k: int = 5, 
                        school_id: str = None, section: str = None) -> list:
        """Perform semantic search with optional filtering"""
        # If filters are provided, search within filtered subset
        if school_id or section:
            filtered_docs = self._filter_docs(school_id, section)
            if not filtered_docs:
                return []
            
            # Create temporary vector store from filtered docs
            from langchain_community.vectorstores import FAISS
            temp_store = FAISS.from_documents(filtered_docs, self.embedding)
            temp_retriever = temp_store.as_retriever(search_kwargs={"k": min(k, len(filtered_docs))})
            results = temp_retriever.invoke(query)
        else:
            # Search in all documents
            self.retriever.search_kwargs["k"] = k
            results = self.retriever.invoke(query)
        
        return results
    
    def retrieve(self, keywords: list[dict], question: str = None) -> tuple[list[WebSource], list[RagSource]]:
        """
        Retrieve documents using RAG with semantic search.
        
        Args:
            keywords: List of keyword dicts with optional school_id, section, query
            question: Optional question string for semantic search
        """
        web_sources: list[WebSource] = []
        rag_sources: list[RagSource] = []
        
        try:
            for idx, kw in enumerate(keywords):
                school_id = kw.get("school_id")
                section = kw.get("section")
                query_text = kw.get("query", "")
                
                # Use question if provided, otherwise use query from keywords
                search_query = question if question else query_text
                if not search_query:
                    # Fallback: construct query from school_id and section
                    search_query = f"{school_id} {section}" if school_id and section else ""
                
                if not search_query.strip():
                    continue
                
                # Perform semantic search
                k_docs = 5  # Retrieve top 5 most relevant documents
                retrieved_docs = self._semantic_search(
                    search_query, 
                    k=k_docs,
                    school_id=school_id,
                    section=section
                )
                
                if not retrieved_docs:
                    print(f"[LocalRetriever] No documents found for query: {search_query}")
                    continue
                
                # Combine retrieved documents
                combined_content = "\n\n".join([doc.page_content for doc in retrieved_docs])
                
                # Get metadata from first document
                first_doc = retrieved_docs[0]
                doc_school_id = first_doc.metadata.get("school_id", school_id or "N/A")
                doc_section = first_doc.metadata.get("section", section or "N/A")
                safe_school_id = str(doc_school_id or school_id or "local").strip().replace(" ", "_")
                safe_section = str(doc_section or section or "general").strip().replace(" ", "_")
                local_source_url = f"localdb://{safe_school_id}/{safe_section}"
                
                title = f"Thông tin trường ĐH-CĐ - {doc_school_id}"
                if doc_section and doc_section != "N/A":
                    title += f" ({doc_section})"
                
                description = combined_content[:200] + "..." if len(combined_content) > 200 else combined_content
                
                # Create WebSource
                web_source: WebSource = {
                    "query": search_query,
                    "title": title,
                    "description": description,
                    "url": local_source_url,
                    "text": combined_content,
                    "files": [],
                    "score": 1.0  # Semantic search results are already ranked
                }
                web_sources.append(web_source)
                
                # Create RAG sources from individual documents
                for chunk_idx, doc in enumerate(retrieved_docs):
                    rag_source: RagSource = {
                        "chunk_index": idx * 1000 + chunk_idx,  # Unique chunk index
                        "query": search_query,
                        "title": title,
                        "url": local_source_url,
                        "text": doc.page_content,
                    }
                    rag_sources.append(rag_source)
                
                print(f"[LocalRetriever] Retrieved {len(retrieved_docs)} documents for query: {search_query}")
        
        except Exception as e:
            print(f"[LocalRetriever] Error during retrieval: {e}")
            traceback.print_exc()
        return web_sources, rag_sources

In [16]:
class WebRetriever:
    """Search in web"""
    def __init__(self, llm_ranker: PageRerankModelProtocol, llm_keywords: KeywordModelProtocol) -> None:
        self.pipeline = DataRetrieverPipeline(
            llm_ranker,
            websearch_config=search_config,
            rag_config=rag_config,
            splitter_config=splitter_config,
            neighbor_merge_config=neighbor_config,
            table_merge_config=table_merge_config
        )
        self.llm_keywords = llm_keywords
        self.school_mapper = SchoolMapper(f"{BASE_PATH}files/school_name.json")

    async def start(self):
        """Initialize websearch"""
        await self.pipeline.start()

    def _tokenize(self, text: str) -> set[str]:
        import re

        return set(re.findall(r"\w+", (text or "").lower(), flags=re.UNICODE))

    def _domain_trust(self, url: str) -> float:
        from urllib.parse import urlparse

        domain = (urlparse(url).netloc or "").lower()
        if not domain:
            return 0.0
        if domain.endswith(".edu.vn") or domain.endswith(".gov.vn"):
            return 1.0
        if ".edu" in domain or ".gov" in domain:
            return 0.9
        if domain.endswith(".org"):
            return 0.75
        if any(bad in domain for bad in ["forum", "blogspot", "wordpress"]):
            return 0.35
        return 0.6

    def _relevance_score(self, question: str, title: str, description: str, url: str) -> float:
        query_tokens = self._tokenize(question)
        text_tokens = self._tokenize(f"{title} {description} {url}")
        if not query_tokens:
            return 0.0
        overlap = len(query_tokens.intersection(text_tokens))
        return overlap / max(1, len(query_tokens))

    def _extract_years(self, text: str) -> list[int]:
        import re

        years = [int(y) for y in re.findall(r"(?<!\d)(19\d{2}|20\d{2})(?!\d)", text or "")]
        return [y for y in years if 1990 <= y <= 2100]

    def _freshness_score(self, question: str, title: str, description: str, url: str) -> float:
        from datetime import datetime

        current_year = datetime.now().year
        question_years = self._extract_years(question)
        doc_years = self._extract_years(f"{title} {description} {url}")

        if not doc_years:
            return 0.50

        if question_years:
            target_year = max(question_years)
            distance = min(abs(y - target_year) for y in doc_years)
            return max(0.0, 1.0 - distance / 5.0)

        newest_doc_year = max(doc_years)
        distance = max(0, current_year - newest_doc_year)
        return max(0.0, 1.0 - distance / 8.0)

    def _richness_score(self, description: str, text: str) -> float:
        merged = f"{description or ''} {text or ''}".strip()
        token_count = len(self._tokenize(merged))
        if token_count <= 5:
            return 0.20
        if token_count <= 15:
            return 0.45
        if token_count <= 30:
            return 0.70
        return 1.00

    def _spam_penalty(self, title: str, description: str, url: str) -> float:
        text = f"{title} {description} {url}".lower()
        bad_terms = ["click", "mua ngay", "khuyen mai", "casino", "cacuoc", "bet", "slot"]
        hits = sum(1 for term in bad_terms if term in text)
        return min(0.5, hits * 0.12)

    def _canonical_url(self, url: str) -> str:
        from urllib.parse import urlparse, parse_qsl, urlencode, urlunparse

        if not url:
            return ""

        parsed = urlparse(url)
        filtered_query = [(k, v) for k, v in parse_qsl(parsed.query, keep_blank_values=False) if not k.lower().startswith("utm_")]
        normalized_path = parsed.path.rstrip("/") or "/"
        normalized = parsed._replace(path=normalized_path, query=urlencode(filtered_query), fragment="")
        return urlunparse(normalized)


    def _source_safety_reason(self, question: str, source: dict) -> str | None:
        from urllib.parse import urlparse
        title = source.get("title", "") or ""
        url = source.get("url", "") or ""
        description = source.get("description", "") or ""
        text = source.get("text", "") or ""
        domain = (urlparse(url).netloc or "").lower()
        merged = _normalize_for_match(f"{title} {description} {url} {text[:3000]}")
        question_norm = _normalize_for_match(question)
        hard_bad_domains = ["vieclam", "topcv", "career", "jobs", "jobstreet", "timviec", "viec-lam", "vieclamtot", "123job", "itviec"]
        job_terms = ["mong mu?n t?m vi?c", "t?m vi?c", "vi?c l?m", "tuy?n d?ng", "nh?n vi?n", "th?c t?p sinh", "tr??ng ph?ng nh?n s?", "e-commerce executive"]
        admission_terms = ["?i?m chu?n", "h?c ph?", "tuy?n sinh", "x?t tuy?n", "ng?nh ??o t?o", "admission", "tuition"]
        asks_admission = any(term in question_norm for term in admission_terms)
        if asks_admission and any(bad in domain for bad in hard_bad_domains):
            return f"bad_domain_for_admission:{domain}"
        if asks_admission and sum(1 for term in job_terms if term in merged) >= 2:
            return "job_content_for_admission_query"
        if len(self._tokenize(text)) < 20 and not source.get("files"):
            return "too_little_content"
        return None

    def _source_safety_filter(self, question: str, web_sources: list[WebSource], rag_sources: list[RagSource], params: GenerationParams) -> tuple[list[WebSource], list[RagSource]]:
        quality_log = params.get("quality_log", False)
        accepted_web, accepted_urls = [], set()
        for idx, source in enumerate(web_sources, 1):
            reason = self._source_safety_reason(question, source)
            if reason:
                if quality_log:
                    print(f"[SourceSafetyGate] [{idx:02d}] DROP | reason={reason} | title={source.get('title', '')[:80]} | url={source.get('url', '')}")
                continue
            accepted_web.append(source)
            canonical = self._canonical_url(source.get("url", ""))
            if canonical:
                accepted_urls.add(canonical)
        accepted_rag = []
        for source in rag_sources:
            canonical = self._canonical_url(source.get("url", ""))
            if not web_sources or (canonical and canonical in accepted_urls):
                accepted_rag.append(source)
        return accepted_web, accepted_rag

    def _quality_audit(
        self,
        question: str,
        web_sources: list[WebSource],
        rag_sources: list[RagSource],
        params: GenerationParams
    ) -> tuple[list[WebSource], list[RagSource]]:
        quality_log = params.get("quality_log", True)

        min_score = float(params.get("quality_min_score", 0.58))
        min_relevance = float(params.get("quality_min_relevance", 0.25))
        min_trust = float(params.get("quality_min_trust", 0.50))
        strict_mode = bool(params.get("quality_strict_mode", True))

        max_docs_after_gate = int(params.get("quality_max_docs", params.get("k_pages", 3)))

        if quality_log:
            print("\n[DocumentQualityGate] STEP 3/3 - Validate and score web documents")
            print(
                f"[DocumentQualityGate] Thresholds -> final_score>={min_score:.2f}, "
                f"relevance>={min_relevance:.2f}, trust>={min_trust:.2f}, strict_mode={strict_mode}"
            )

        accepted: list[tuple[float, WebSource, str]] = []
        seen_urls: set[str] = set()

        for idx, source in enumerate(web_sources, 1):
            title = source.get("title", "")
            url = source.get("url", "")
            description = source.get("description", source.get("text", ""))
            source_text = source.get("text", "")

            canonical_url = self._canonical_url(url)
            safety_reason = self._source_safety_reason(question, source)
            if safety_reason:
                if quality_log:
                    print(f"[DocumentQualityGate] [{idx:02d}] DROP | reason={safety_reason} | title={title[:80]} | url={url}")
                continue
            if canonical_url and canonical_url in seen_urls:
                if quality_log:
                    print(f"[DocumentQualityGate] [{idx:02d}] DROP | reason=duplicate_url | url={url}")
                continue

            relevance = self._relevance_score(question, title, description, url)
            trust = self._domain_trust(url)
            freshness = self._freshness_score(question, title, description, url)
            richness = self._richness_score(description, source_text)
            spam_penalty = self._spam_penalty(title, description, url)

            final_score = (
                0.45 * relevance
                + 0.25 * trust
                + 0.20 * freshness
                + 0.10 * richness
                - spam_penalty
            )
            final_score = max(0.0, min(1.0, final_score))

            if strict_mode:
                is_ok = final_score >= min_score and relevance >= min_relevance and trust >= min_trust
            else:
                is_ok = final_score >= min_score

            if quality_log:
                status = "PASS" if is_ok else "DROP"
                print(
                    f"[DocumentQualityGate] [{idx:02d}] {status} | final={final_score:.3f} "
                    f"(rel={relevance:.3f}, trust={trust:.3f}, fresh={freshness:.3f}, rich={richness:.3f}, spam={spam_penalty:.3f}) | "
                    f"title={title[:80]} | url={url}"
                )

            if is_ok:
                accepted.append((final_score, source, canonical_url))
                if canonical_url:
                    seen_urls.add(canonical_url)

        accepted.sort(key=lambda x: x[0], reverse=True)
        accepted = accepted[:max(0, max_docs_after_gate)]

        accepted_web_sources: list[WebSource] = [item[1] for item in accepted]
        accepted_urls: set[str] = {item[2] for item in accepted if item[2]}

        accepted_rag_sources: list[RagSource] = []
        for source in rag_sources:
            source_url = source.get("url", "")
            source_canonical = self._canonical_url(source_url)
            if (not web_sources and not accepted_urls) or (source_canonical and source_canonical in accepted_urls):
                accepted_rag_sources.append(source)

        if quality_log:
            print(
                f"[DocumentQualityGate] Result -> web_sources={len(accepted_web_sources)}/{len(web_sources)}, "
                f"rag_sources={len(accepted_rag_sources)}/{len(rag_sources)}, "
                f"max_docs_after_gate={max_docs_after_gate}"
            )

        return accepted_web_sources, accepted_rag_sources

    async def retrive(self, question: str, params: GenerationParams) -> tuple[list[WebSource], list[RagSource]]:
        enable_quality_gate = params.get("enable_quality_gate", False)
        rich_media_enabled = bool(params.get("include_pdf") or params.get("include_image"))
        if rich_media_enabled and params.get("quality_force_for_rich_media", True):
            if not enable_quality_gate:
                print("[DocumentQualityGate] Forced ON because include_pdf/include_image is enabled")
            enable_quality_gate = True
            params["enable_quality_gate"] = True
        quality_log = params.get("quality_log", enable_quality_gate)
        params["quality_log"] = quality_log

        if quality_log:
            print("\n[DocumentQualityGate] STEP 1/3 - Send query to keyword extractor")
            print(f"[DocumentQualityGate] Question: {question}")

        data = await self.llm_keywords.keywords(question, params)
        max_query = params.get("max_query", 1)
        queries = []
        school_restrict = params.get("school_domain", False)

        for item in data:
            if not school_restrict:
                queries.append(item["query"])
            else:
                school = item["school"]
                if school.strip() != "":
                    school_domains = self.school_mapper.domains_from_auto(school, 5)[:10]
                    print(f"[DOMAINS]", school_domains)
                    if len(school_domains) > 0:
                        queries.append([item["query"], school_domains])

        queries = queries[:max_query]
        if quality_log:
            print(f"[DocumentQualityGate] Extracted queries ({len(queries)}): {queries}")
            print("[DocumentQualityGate] STEP 2/3 - Run web search")

        web_sources, rag_sources = await self.pipeline.retrieve(params, queries)

        if quality_log:
            print(
                f"[DocumentQualityGate] Raw retrieve -> web_sources={len(web_sources)}, "
                f"rag_sources={len(rag_sources)}"
            )

        if not enable_quality_gate:
            if quality_log:
                print("[DocumentQualityGate] Disabled -> return raw web search output")
            if params.get("source_safety_filter", True):
                return self._source_safety_filter(question, web_sources, rag_sources, params)
            return web_sources, rag_sources

        return self._quality_audit(question, web_sources, rag_sources, params)

In [ ]:
# Updated RouterRetriever with chunk rerank evaluation, fallback, and LLM time detection
TIME_DETECT_INSTRUCTION = """Bạn là trợ lý trích xuất khoảng năm tìm kiếm. Trả JSON duy nhất: {\"metric\": \"y|none\", \"year_start\": int|null, \"year_end\": int|null}.
- Nếu không thấy năm: metric=none, year_start=null, year_end=null.
- Nếu một năm (vd 2020): metric=y, year_start=2020, year_end=2020.
- Nếu khoảng (vd 2020-2022): metric=y, year_start=2020, year_end=2022.
"""
TIME_DETECT_TEMPLATE = """Câu hỏi: {question}\nTrả lời đúng JSON duy nhất."""

class RouterRetriever:
    def __init__(self, llm_router: RouterModelProtocol, web_retriever: WebRetriever, local_retriever: LocalRetriever) -> None:
        self.web_retriever = web_retriever
        self.local_retriever = local_retriever
        self.router = llm_router
        # Initialize chunk ranker for evaluating local results
        from data_retriever.ranker import ChunkRanker
        from data_retriever.config import ChunkRankerConfig
        chunk_ranker_config = ChunkRankerConfig()
        self.chunk_ranker = ChunkRanker(chunk_ranker_config)
        # Threshold for local result quality
        self.local_min_score = 0.5  # Minimum max score to consider local results good
        self.local_min_chunks = 1   # Minimum number of chunks after reranking
    
    def _evaluate_local_results(self, rag_sources: list[RagSource], question: str) -> tuple[bool, float, int]:
        """
        Evaluate local results using chunk reranker.
        Returns: (is_sufficient, max_score, num_valid_chunks)
        """
        if not rag_sources:
            return False, 0.0, 0
        
        # Use chunk reranker to evaluate and filter chunks
        reranked_sources = self.chunk_ranker.rerank_chunks(rag_sources, question, relative_threshold=0.3)
        
        if not reranked_sources:
            return False, 0.0, 0
        
        num_valid = len(reranked_sources)
        # Check if results are sufficient
        is_sufficient = num_valid >= self.local_min_chunks
        
        print(f"[RouterRetriever] Local evaluation: {num_valid} valid chunks, sufficient={is_sufficient}")
        
        return is_sufficient, 1.0, num_valid

    async def _detect_time_with_llm(self, question: str, params: GenerationParams) -> None:
        """Use LLM to infer time_metric/time_range if user didn't set."""
        if params.get("time_metric") or params.get("time_range"):
            return
        prompt = TIME_DETECT_TEMPLATE.format(question=question)
        copy_params = copy.deepcopy(params)
        copy_params.update({
            "max_tokens": 128,
            "temperature": 0.0,
            "top_p": 0.9,
        })
        text = ""
        try:
            async for chunk in await self.router(
                call_type=CallType.KEYWORDS,
                instruction=TIME_DETECT_INSTRUCTION,
                prompt=prompt,
                params=copy_params
            ):
                text += chunk
        except Exception:
            traceback.print_exc()
            return
        data = None
        try:
            from backend.route.utils import extract_json
            data = json.loads(extract_json(text))
        except Exception:
            try:
                start = text.find("{")
                end = text.rfind("}") + 1
                if start != -1 and end > start:
                    data = json.loads(text[start:end])
            except Exception:
                traceback.print_exc()
                return
        if not isinstance(data, dict):
            return
        metric = data.get("metric")
        year_start = data.get("year_start")
        year_end = data.get("year_end")
        if metric == "y" and isinstance(year_start, int) and isinstance(year_end, int):
            if 1900 <= year_start <= 2100 and 1900 <= year_end <= 2100:
                if year_start > year_end:
                    year_start, year_end = year_end, year_start
                params["time_metric"] = "y"
                params.pop("time_range", None)
                params.pop("time_year", None)
                params["time_year_start"] = year_start
                params["time_year_end"] = year_end
                print(f"[RouterRetriever][TIME LLM] year_start={year_start} year_end={year_end} metric=y")
        elif metric == "y" and isinstance(year_start, int) and 1900 <= year_start <= 2100:
            params["time_metric"] = "y"
            params.pop("time_range", None)
            params.pop("time_year", None)
            params["time_year_start"] = year_start
            params["time_year_end"] = year_start
            print(f"[RouterRetriever][TIME LLM] year={year_start} metric=y")
        elif metric in ["y", "m", "d"] and data.get("range"):
            time_range = data.get("range")
            if isinstance(time_range, int) and time_range > 0:
                params["time_metric"] = metric
                params["time_range"] = min(time_range, 365)
                print(f"[RouterRetriever][TIME LLM] metric={metric} range={params['time_range']}")
    
    async def retrieve(self, question: str, params: GenerationParams) -> tuple[list[WebSource], list[RagSource]]:
        use_websearch = params.get("use_websearch", False) and params.get("max_query", 0) > 0 and params.get("k_docs", 0) > 0 and params.get("k_pages", 0) > 0
        use_localdb = params.get("use_localdb", False)
        
        if use_websearch:
            await self._detect_time_with_llm(question, params)
        
        if use_websearch and use_localdb:
            hybrid_retrieval = bool(params.get("hybrid_retrieval", False))
            local_queries = await self.router.route(question, params)
            if len(local_queries) > 0:
                # Pass question to LocalRetriever for semantic search
                web_sources, rag_sources = self.local_retriever.retrieve(local_queries, question=question)
                # Evaluate local results
                is_sufficient, max_score, num_chunks = self._evaluate_local_results(rag_sources, question)
                
                if is_sufficient:
                    if hybrid_retrieval:
                        print(f"[RouterRetriever] Hybrid retrieval: merge local ({num_chunks} chunks) and web results")
                        web_web, web_rag = await self.web_retriever.retrive(question, params)
                        return web_sources + web_web, rag_sources + web_rag
                    print(f"[RouterRetriever] Using local results ({num_chunks} chunks)")
                    return web_sources, rag_sources
                else:
                    print(f"[RouterRetriever] Local results insufficient ({num_chunks} chunks, score={max_score:.3f}), falling back to websearch")
                    return await self.web_retriever.retrive(question, params)
            else:
                return await self.web_retriever.retrive(question, params)
        elif use_localdb:
            local_queries = await self.router.route(question, params)
            if len(local_queries) > 0:
                # Pass question to LocalRetriever for semantic search
                web_sources, rag_sources = self.local_retriever.retrieve(local_queries, question=question)
                # Evaluate local results
                is_sufficient, max_score, num_chunks = self._evaluate_local_results(rag_sources, question)
                
                if is_sufficient:
                    print(f"[RouterRetriever] Using local results ({num_chunks} chunks)")
                    return web_sources, rag_sources
                else:
                    print(f"[RouterRetriever] Local results insufficient ({num_chunks} chunks, score={max_score:.3f})")
                    # If websearch is available, fallback
                    if use_websearch:
                        print(f"[RouterRetriever] Falling back to websearch")
                        return await self.web_retriever.retrive(question, params)
                    else:
                        return web_sources, rag_sources  # Return what we have even if insufficient
            else:
                return [], []
        elif use_websearch:
            return await self.web_retriever.retrive(question, params)
        else:
            return [], []


##### Client to call model

In [18]:
from vllm import SamplingParams, AsyncLLMEngine, AsyncEngineArgs
from vllm.outputs import RequestOutput
from vllm.utils import random_uuid
from vllm.lora.request import LoRARequest
from typing import AsyncGenerator
from typing import Optional, Any
from vllm.transformers_utils.tokenizers import MistralTokenizer
from openai.types.chat import  ChatCompletionUserMessageParam, ChatCompletionSystemMessageParam
from vllm.entrypoints.chat_utils import (
    ChatTemplateContentFormatOption, 
    resolve_chat_template_content_format, 
    apply_hf_chat_template,
    apply_mistral_chat_template,
    parse_chat_messages
)
from vllm.inputs.data import TokensPrompt

class AsyncLLMEngineWrapper:
    """Not support shutdown"""
    def __init__(self) -> None:
        self.engine = None
        self.reap_wait_time = 5
    def init(self, engine_args: AsyncEngineArgs):
        self.engine = AsyncLLMEngine.from_engine_args(engine_args)
    def generate(self, prompt: str | TokensPrompt, sampling_params: SamplingParams, lora_request: LoRARequest | None) -> AsyncGenerator[RequestOutput, None]:
        if self.engine is None:
            raise Exception("Not initialized")
        return self.engine.generate(
            prompt=prompt,
            sampling_params=sampling_params,
            request_id=random_uuid(),
            lora_request=lora_request
        )
    async def chat(self, instruction: str, prompt: str, sampling_params: SamplingParams, lora_request: LoRARequest | None = None):
        messages = [
            ChatCompletionSystemMessageParam(content=instruction, role="system"),
            ChatCompletionUserMessageParam(content=prompt, role="user")
        ]
        return await self._chat(
            messages=messages,
            sampling_params=sampling_params,
            lora_request=lora_request,
            chat_template_kwargs={
                "enable_thinking": False
            }
        )
    async def _chat(
        self,
        messages: list[ChatCompletionUserMessageParam | ChatCompletionUserMessageParam],
        sampling_params: SamplingParams,
        lora_request: LoRARequest | None,
        chat_template_content_format: ChatTemplateContentFormatOption = "auto",
        chat_template: Optional[str] = None,
        add_generation_prompt: bool = True,
        continue_final_message: bool = False,
        chat_template_kwargs: Optional[dict[str, Any]] = None
    ):
        if self.engine is None: raise Exception("Model not loaded")
        tokenizer = await self.engine.get_tokenizer(lora_request)
        model_config = self.engine.engine.get_model_config()
        resolved_content_format = resolve_chat_template_content_format(
            chat_template,
            None,
            chat_template_content_format,
            tokenizer,
            model_config=model_config,
        )
        _chat_template_kwargs: dict[str, Any] = dict(
            chat_template=chat_template,
            add_generation_prompt=add_generation_prompt,
            continue_final_message=continue_final_message,
            tools=None,
        )
        _chat_template_kwargs.update(chat_template_kwargs or {})
        conversation, _ = parse_chat_messages(
            messages, #type:ignore
            model_config,
            tokenizer,
            content_format=resolved_content_format,
        )

        if isinstance(tokenizer, MistralTokenizer):
            prompt_token_ids = apply_mistral_chat_template(
                tokenizer,
                messages=messages, #type:ignore
                **_chat_template_kwargs,
            )
        else:
            prompt_str = apply_hf_chat_template(
                tokenizer=tokenizer, #type:ignore
                conversation=conversation,
                model_config=model_config,
                **_chat_template_kwargs,
            )
            prompt_token_ids = tokenizer.encode(prompt_str, add_special_tokens=False)
        prompt = TokensPrompt(prompt_token_ids=prompt_token_ids)
        return self.generate(
            prompt,
            sampling_params=sampling_params,
            lora_request=lora_request,
        )

INFO 12-02 16:08:03 [__init__.py:235] Automatically detected platform cuda.


In [ ]:
import os
import json
import msgspec
import glob

_DEBUG_FALSE_VALUES = {"0", "false", "no", "off"}


def _debug_trace_enabled() -> bool:
    return str(os.getenv("BOT_DEBUG_TRACE", "1")).strip().lower() not in _DEBUG_FALSE_VALUES


def _debug_full_prompt_enabled() -> bool:
    return str(os.getenv("BOT_DEBUG_FULL_PROMPTS", "0")).strip().lower() not in _DEBUG_FALSE_VALUES


def _debug_int_env(name: str, default: int) -> int:
    try:
        return int(os.getenv(name, str(default)))
    except Exception:
        return default


def _debug_clip(text, limit: int | None = None) -> str:
    value = "" if text is None else str(text)
    if limit is None:
        limit = _debug_int_env("BOT_DEBUG_TEXT_CHARS", 12000)
    if limit <= 0 or len(value) <= limit:
        return value
    return value[:limit] + f"\n...[truncated {len(value) - limit} chars]"


def _debug_redact(value):
    secret_markers = ("KEY", "TOKEN", "SECRET", "PASSWORD", "AUTH")
    if isinstance(value, dict):
        out = {}
        for key, item in value.items():
            key_str = str(key)
            if any(marker in key_str.upper() for marker in secret_markers):
                out[key_str] = "[REDACTED]"
            else:
                out[key_str] = _debug_redact(item)
        return out
    if isinstance(value, (list, tuple)):
        return [_debug_redact(item) for item in value]
    return value


def _debug_json(value) -> str:
    try:
        return json.dumps(_debug_redact(value), ensure_ascii=False, indent=2, default=str)
    except Exception:
        return str(value)


def _debug_block(title: str, body="", *, limit: int | None = None) -> None:
    if not _debug_trace_enabled():
        return
    print("\n" + "=" * 80)
    print(f"[{title}]")
    print("=" * 80)
    if body is not None:
        print(_debug_clip(body, limit))


def _debug_source_list(label: str, sources: list) -> None:
    if not _debug_trace_enabled():
        return
    text_limit = _debug_int_env("BOT_DEBUG_SOURCE_TEXT_CHARS", 0)
    print("\n" + "=" * 80)
    print(f"[{label}] count={len(sources)} text_limit={text_limit} (0=full)")
    print("=" * 80)
    if not sources:
        print("(empty)")
        return
    for idx, source in enumerate(sources, 1):
        text = source.get("text", "") if isinstance(source, dict) else ""
        meta = {
            "index": idx,
            "title": source.get("title", "") if isinstance(source, dict) else "",
            "url": source.get("url", "") if isinstance(source, dict) else "",
            "query": source.get("query", "") if isinstance(source, dict) else "",
            "score": source.get("score", None) if isinstance(source, dict) else None,
            "chunk_index": source.get("chunk_index", None) if isinstance(source, dict) else None,
            "text_chars": len(text or ""),
        }
        print("-" * 80)
        print(_debug_json(meta))
        print("[SOURCE_TEXT]")
        print(_debug_clip(text, text_limit))


def _strip_accents_for_match(text: str) -> str:
    import unicodedata
    normalized = unicodedata.normalize("NFD", text or "")
    without_marks = "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")
    return without_marks.replace("đ", "d").replace("Đ", "D")


def _normalize_for_match(text: str) -> str:
    import re
    return re.sub(r"\s+", " ", _strip_accents_for_match(text).lower()).strip()


def _term_in_text(term: str, text: str) -> bool:
    import re
    term_norm = _normalize_for_match(term)
    if not term_norm:
        return False
    if re.fullmatch(r"[a-z0-9\s.+/#&-]+", term_norm):
        return re.search(rf"(?<!\w){re.escape(term_norm)}(?!\w)", text) is not None
    return term_norm in text


def _query_focus_terms(question: str) -> list[str]:
    import re
    text = _normalize_for_match(question)
    terms = []
    signal_terms = [
        "tuyen sinh", "de an tuyen sinh", "thong tin tuyen sinh",
        "diem chuan", "diem trung tuyen", "diem xet tuyen", "diem san",
        "diem nhan ho so", "nguong dau vao", "nguong dam bao chat luong",
        "hoc phi", "le phi", "hoc bong", "mien giam", "tin dung sinh vien",
        "chi tieu", "ma nganh", "ten nganh", "to hop mon", "khoi thi",
        "phuong thuc xet tuyen", "xet tuyen", "xet tuyen som", "xet tuyen ket hop",
        "tuyen thang", "uu tien xet tuyen", "nguyen vong", "ho so",
        "thoi gian dang ky", "han dang ky", "lich tuyen sinh", "nhap hoc",
        "xac nhan nhap hoc", "cong bo ket qua",
        "hoc ba", "thi thpt", "tot nghiep thpt", "danh gia nang luc",
        "danh gia tu duy", "dgnl", "dgtd", "tsa", "sat", "ielts", "toefl",
        "chuong trinh dao tao", "chuan dau ra", "thoi gian dao tao",
        "bang cap", "cu nhan", "ky su", "chat luong cao", "tien tien",
        "lien ket quoc te", "song bang",
        "ky tuc xa", "noi tru", "ngoai tru", "co so dao tao", "dia diem hoc",
        "campus", "co so vat chat",
        "co hoi viec lam", "viec lam", "muc luong", "thuc tap",
        "doanh nghiep", "dau ra",
    ]
    terms.extend(term for term in signal_terms if _term_in_text(term, text))
    phrase_patterns = [
        r"\b(?:nganh|nhom nganh|linh vuc|chuyen nganh|chuong trinh|chuong trinh dao tao|khoa)\s+([a-z0-9][a-z0-9\s/&+.\-]{2,80})",
        r"\b(?:truong|dai hoc|hoc vien)\s+([a-z0-9][a-z0-9\s/&+.\-]{2,80})",
    ]
    stop_tail = re.compile(
        r"\b(?:nam|tai|o|cua|cac|nhung|co|khong|la|bao nhieu|cao|thap|nhat|kem|voi|va|so sanh|xep hang|top|theo)\b.*$"
    )
    for pattern in phrase_patterns:
        for match in re.finditer(pattern, text):
            phrase = stop_tail.sub("", match.group(1)).strip(" ,.;:-")
            if len(phrase) >= 3:
                terms.append(phrase)
    for abbr in re.findall(r"\b[A-Z0-9][A-Z0-9+/&.-]{1,8}\b", question or ""):
        terms.append(_normalize_for_match(abbr))
    domain_aliases = {
        "cong nghe thong tin": ["cntt", "it", "information technology"],
        "khoa hoc may tinh": ["khmt", "computer science"],
        "ky thuat may tinh": ["computer engineering"],
        "tri tue nhan tao": ["ai", "artificial intelligence"],
        "khoa hoc du lieu": ["khdl", "data science"],
        "an toan thong tin": ["attt", "information security"],
        "an ninh mang": ["cyber security", "cybersecurity"],
        "ky thuat phan mem": ["software engineering", "ktpm"],
        "he thong thong tin": ["httt", "information systems"],
        "mang may tinh": ["computer network", "networking"],
        "thuong mai dien tu": ["tmdt", "ecommerce", "e-commerce"],
        "dien tu vien thong": ["dtvt", "electronics and telecommunications"],
        "ky thuat dien": ["dien dien tu", "electrical engineering"],
        "dieu khien va tu dong hoa": ["automation", "tu dong hoa"],
        "co dien tu": ["mechatronics"],
        "ky thuat co khi": ["co khi", "mechanical engineering"],
        "ky thuat o to": ["cong nghe ky thuat o to", "automotive engineering"],
        "xay dung": ["ky thuat xay dung", "civil engineering"],
        "kien truc": ["architecture"],
        "ky thuat hoa hoc": ["hoa hoc", "chemical engineering"],
        "cong nghe thuc pham": ["food technology"],
        "ky thuat moi truong": ["moi truong", "environmental engineering"],
        "logistics": ["logistics va quan ly chuoi cung ung", "supply chain"],
        "quan tri kinh doanh": ["qtkd", "business administration"],
        "marketing": ["digital marketing"],
        "kinh doanh quoc te": ["international business"],
        "tai chinh ngan hang": ["finance banking", "ngan hang"],
        "ke toan": ["accounting"],
        "kiem toan": ["auditing"],
        "ngon ngu anh": ["english language"],
        "ngon ngu trung": ["tieng trung", "chinese language"],
        "ngon ngu nhat": ["tieng nhat", "japanese language"],
        "luat": ["law"],
        "luat kinh te": ["economic law"],
        "quan he quoc te": ["international relations"],
        "bao chi": ["journalism"],
        "truyen thong da phuong tien": ["multimedia communication"],
        "tam ly hoc": ["psychology"],
        "su pham": ["giao duc", "teacher education"],
        "y khoa": ["medicine", "bac si da khoa"],
        "duoc hoc": ["pharmacy"],
        "dieu duong": ["nursing"],
        "rang ham mat": ["dentistry"],
        "y hoc co truyen": ["traditional medicine"],
        "xet nghiem y hoc": ["medical laboratory"],
        "thu y": ["veterinary"],
        "nong nghiep": ["agriculture"],
        "thuy san": ["aquaculture"],
        "thiet ke do hoa": ["graphic design"],
        "my thuat": ["fine arts"],
    }
    for canonical, aliases in domain_aliases.items():
        if _term_in_text(canonical, text) or any(_term_in_text(alias, text) for alias in aliases):
            terms.extend([canonical, *aliases])
    terms.extend(re.findall(r"\b(?:19|20)\d{2}\b", question or ""))
    out, seen = [], set()
    for term in terms:
        norm = _normalize_for_match(term)
        if norm and norm not in seen:
            seen.add(norm)
            out.append(norm)
    return out


def _select_relevant_evidence_text(question: str, text: str, max_chars: int) -> str:
    import re
    if not text or max_chars <= 0 or len(text) <= max_chars:
        return text or ""
    focus_terms = _query_focus_terms(question)
    q_tokens = set(re.findall(r"\w+", _normalize_for_match(question), flags=re.UNICODE))
    lines = [ln.strip() for ln in (text or "").splitlines() if ln.strip()]
    if not lines:
        return text[:max_chars]
    selected_idx, scored = set(), []
    for idx, line in enumerate(lines):
        norm = _normalize_for_match(line)
        score = 0
        if "|" in line and any(h in norm for h in [
            "ma nganh", "ten nganh", "2024", "2025", "hoc phi", "diem chuan",
            "diem trung tuyen", "diem san", "chi tieu", "to hop", "khoi thi",
            "phuong thuc", "xet tuyen", "hoc bong", "ma xet tuyen",
        ]):
            score += 4
            selected_idx.add(idx)
            if idx + 1 < len(lines):
                selected_idx.add(idx + 1)
        score += sum(5 for term in focus_terms if term and term in norm)
        line_tokens = set(re.findall(r"\w+", norm, flags=re.UNICODE))
        score += min(5, len(q_tokens.intersection(line_tokens)))
        if score > 0:
            scored.append((score, idx))
    for _, idx in sorted(scored, key=lambda item: (-item[0], item[1])):
        for j in range(max(0, idx - 1), min(len(lines), idx + 2)):
            selected_idx.add(j)
        candidate = "\n".join(lines[j] for j in sorted(selected_idx))
        if len(candidate) >= max_chars:
            break
    selected = "\n".join(lines[j] for j in sorted(selected_idx))
    if not selected.strip():
        selected = text[:max_chars]
    return selected[:max_chars]


def _is_table_like_text(text: str) -> bool:
    return "[BANG]" in (text or "") or (text or "").count("|") >= 6


def _extract_admission_table_evidence(question: str, text: str, max_chars: int = 6000) -> str:
    if not text or max_chars <= 0 or not _is_table_like_text(text):
        return ""
    q_norm = _normalize_for_match(question)
    score_query = any(
        _term_in_text(term, q_norm)
        for term in ["diem chuan", "diem trung tuyen", "diem xet tuyen", "diem san"]
    )
    if not score_query:
        return ""
    lines = [ln.strip() for ln in (text or "").splitlines() if ln.strip()]
    if not lines:
        return ""
    header_terms = [
        "diem chuan", "diem trung tuyen", "diem xet tuyen", "diem san",
        "ma xet tuyen", "ma nganh", "ten nganh", "nganh dao tao", "to hop",
    ]
    start_idx = None
    for idx, line in enumerate(lines):
        norm = _normalize_for_match(line)
        is_table_marker = "[bang]" in norm
        is_header = "|" in line and any(_term_in_text(term, norm) for term in header_terms)
        if is_table_marker or is_header:
            start_idx = max(0, idx - 4)
            break
    if start_idx is None:
        table_rows = [i for i, line in enumerate(lines) if "|" in line and line.count("|") >= 2]
        if len(table_rows) < 3:
            return ""
        start_idx = max(0, table_rows[0] - 3)
    selected = []
    for line in lines[start_idx:]:
        candidate = "\n".join(selected + [line])
        if len(candidate) > max_chars:
            break
        selected.append(line)
    return "\n".join(selected).strip()


def _augment_rag_with_web_table_evidence(question: str, web_sources: list, rag_sources: list, *, max_sources: int = 2, max_chars: int = 6000) -> list:
    if not web_sources:
        return rag_sources
    existing_table_urls = {r.get("url", "") for r in rag_sources if _is_table_like_text(r.get("text", "") or "")}
    additions = []
    for idx, source in enumerate(web_sources, 1):
        url = source.get("url", "")
        if not url or url in existing_table_urls:
            continue
        table_text = _extract_admission_table_evidence(question, source.get("text", "") or "", max_chars=max_chars)
        if not table_text:
            continue
        additions.append({
            "chunk_index": -100000 - idx,
            "query": question,
            "title": source.get("title", ""),
            "url": url,
            "text": table_text,
        })
        if len(additions) >= max_sources:
            break
    if additions:
        print(f"[ReaderContextRescue] Added {len(additions)} table chunk(s) from web_sources because selected RAG missed admissions tables")
        _debug_source_list("READER_RESCUED_TABLE_CHUNKS", additions)
    return additions + rag_sources

class VLLMModelCore:
    def __init__(self) -> None:
        self._engine = AsyncLLMEngineWrapper()
        self.logger = CmdLogger("Model")
        # Sử dụng BAAI bge-reranker-v2-m3 (giống ChunkRanker)
        self._reranker_name = os.getenv("PAGE_RERANKER_MODEL", "BAAI/bge-reranker-v2-m3")
        # Giảm batch size và max_length mặc định để đỡ tốn VRAM (có thể override bằng env)
        self._reranker_batch_size = int(os.getenv("PAGE_RERANKER_BATCH_SIZE", "2"))
        self._reranker_max_length = int(os.getenv("PAGE_RERANKER_MAX_LENGTH", "384"))
        self._reranker_device = None
        self._reranker_model = None
    
    @property
    def shared_reranker(self):
        """Get shared reranker instance (for ChunkRanker to reuse)"""
        self._ensure_reranker_loaded()
        return self._reranker_model
    
    @property
    def shared_reranker_device(self):
        """Get shared reranker device"""
        self._ensure_reranker_loaded()
        return self._reranker_device
    
    def init(self, engine_args: AsyncEngineArgs):
        self._engine.init(engine_args)
    async def call(self, call_type: CallType, instruction: str, prompt: str, params: GenerationParams) -> AsyncGenerator[str, None]:
        print(f"[VLLM] {call_type} | Instruction length: {len(instruction)} | Prompt length: {len(prompt)} | kwargs: {params.get('kwargs')}")
        if _debug_trace_enabled():
            prompt_limit = _debug_int_env("BOT_DEBUG_PROMPT_CHARS", 20000)
            _debug_block(
                f"LLM_CALL {call_type}",
                _debug_json({
                    "call_type": str(call_type),
                    "model_id": params.get("model_id"),
                    "max_tokens": params.get("max_tokens"),
                    "temperature": params.get("temperature"),
                    "top_p": params.get("top_p"),
                    "top_k": params.get("top_k"),
                    "params": dict(params),
                    "instruction_chars": len(instruction or ""),
                    "prompt_chars": len(prompt or ""),
                }),
                limit=0,
            )
            if _debug_full_prompt_enabled():
                _debug_block(f"LLM_SYSTEM_PROMPT {call_type}", instruction, limit=prompt_limit)
                _debug_block(f"LLM_USER_PROMPT {call_type}", prompt, limit=prompt_limit)
        model_id = params["model_id"]
        lora_request = None
        if call_type == CallType.READER and SEP in model_id:
            lora_int_id = int(model_id.split(SEP)[-1])
            lora_request = LORA_MAP.get(lora_int_id)
        sampling_params = msgspec.convert(params, SamplingParams)
        if lora_request != None:
            print(f"[VLLm] Using {lora_request.lora_name}")
        stream = await self._engine.chat(
            instruction=instruction,
            prompt=prompt,
            sampling_params=sampling_params,
            lora_request=lora_request
        )
        total_text = ""
        last_index = 0
        async for event in stream:
            total_text = event.outputs[0].text
            yield total_text[last_index:]
            last_index = len(total_text)
    async def __call__(self, call_type: CallType, instruction: str, prompt: str, params: GenerationParams) -> AsyncGenerator[str, None]:
        return self.call(call_type, instruction, prompt, params)

In [ ]:
class VLLMModel(VLLMModelCore):
    async def route(self, question: str, params: GenerationParams) -> list[dict]:
        text = ""
        prompt = ROUTER_TEMPLATE.format(question=question)
        copy_params = copy.deepcopy(params)
        copy_params.update(ROUTER_PARAMS) #type:ignore 
        async for chunk in await self(
            call_type=CallType.ROUTER, 
            instruction=ROUTER_INSTRUCTION, 
            prompt=ROUTER_PREFIX+prompt, 
            params=copy_params
        ):
            text += chunk
        try:
            self.logger.log(text)
            result = json.loads(extract_json(text))
            return result
        except:
            traceback.print_exc()
            return []
    def _ensure_reranker_loaded(self):
        if self._reranker_model is not None and self._reranker_device is not None:
            return
        
        # Auto-detect device: use CPU if CUDA is out of memory, otherwise use configured device
        device_str = "cuda" if torch.cuda.is_available() else "cpu"
        if device_str == "cuda":
            try:
                # Try to allocate a small tensor to check if CUDA has memory
                test_tensor = torch.zeros(1).cuda()
                del test_tensor
                torch.cuda.empty_cache()
            except RuntimeError:
                # CUDA out of memory, fallback to CPU
                device_str = "cpu"
                print(f"[PageReranker] CUDA out of memory, falling back to CPU")
        
        self._reranker_device = device_str
        
        # Load BAAI reranker using CrossEncoder (giống ChunkRanker)
        self._reranker_model = CrossEncoder(
            self._reranker_name,
            max_length=self._reranker_max_length,
            device=device_str
        )
        print(f"[PageReranker] Loaded model: {self._reranker_name} on {device_str}")
    def _make_page_text(self, page: SearchResult) -> str:
        parts = [
            page.get("title", ""),
            page.get("description", ""),
            page.get("url", "")
        ]
        return "\n\n".join([part for part in parts if part])
    async def _llm_rerank_page(self, pages: list[SearchResult], query: str, relative_threshold: float, params: GenerationParams) -> list[SearchResult]:
        if len(pages) == 0:
            return []
        # Dọn bộ nhớ trước khi rerank để giảm nguy cơ OOM trên GPU
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        try:
            self._ensure_reranker_loaded()
        except Exception as e:
            traceback.print_exc()
            raise RuntimeError(f"Failed to load reranker model: {e}") from e
        if self._reranker_model is None or self._reranker_device is None:
            raise RuntimeError("Reranker model or device is not initialized")
        page_texts = [self._make_page_text(page) for page in pages]
        
        # DEBUG: Log reranker input
        print(f"\n[Reranker Debug] Query: {query}")
        print(f"[Reranker Debug] Number of pages: {len(pages)}")
        print(f"[Reranker Debug] Sample page texts (first 3):")
        for i, (page, page_text) in enumerate(zip(pages[:3], page_texts[:3]), 1):
            print(f"\n  Page {i}:")
            print(f"    Title: {page.get('title', 'N/A')[:100]}")
            print(f"    Description: {page.get('description', 'N/A')[:100]}")
            print(f"    URL: {page.get('url', 'N/A')}")
            print(f"    Page text (first 200 chars): {page_text[:200]}...")
        
        def _run_reranker() -> list[float]:
            scores: list[float] = []
            # Prepare pairs for CrossEncoder: [(query, text1), (query, text2), ...]
            pairs = [(query, page_text) for page_text in page_texts]
            
            # Get scores from CrossEncoder (batch processing)
            for start in range(0, len(pairs), self._reranker_batch_size):
                end = start + self._reranker_batch_size
                batch_pairs = pairs[start:end]
                
                # DEBUG: Log first batch
                if start == 0:
                    print(f"\n[Reranker Debug] First batch:")
                    print(f"  Query: {batch_pairs[0][0]}")
                    print(f"  First page text: {batch_pairs[0][1][:300]}...")
                
                # Clear CUDA cache before prediction to avoid OOM
                if self._reranker_device == "cuda" and torch.cuda.is_available():
                    torch.cuda.empty_cache()
                
                # CrossEncoder.predict() returns numpy array
                batch_scores = self._reranker_model.predict(batch_pairs)
                
                # Convert to list of floats
                if hasattr(batch_scores, 'tolist'):
                    batch_scores = batch_scores.tolist()
                elif isinstance(batch_scores, (list, tuple)):
                    batch_scores = [float(s) for s in batch_scores]
                else:
                    batch_scores = [float(batch_scores)]
                
                # DEBUG: Log raw scores
                if start == 0:
                    print(f"\n[Reranker Debug] Raw scores from first batch:")
                    for i, (score, page) in enumerate(zip(batch_scores, pages[start:end]), 1):
                        print(f"  [{i}] Score: {score:.6f} | Title: {page.get('title', 'N/A')[:80]}")
                
                scores.extend(batch_scores)
            
            return scores
        try:
            scores = await asyncio.to_thread(_run_reranker)
        except Exception as e:
            traceback.print_exc()
            raise RuntimeError(f"Failed to run reranker inference: {e}") from e
        if len(scores) != len(pages):
            raise RuntimeError(f"Reranker returned {len(scores)} scores but expected {len(pages)} scores")
        
        # DEBUG: Log all scores before threshold
        print(f"\n[Reranker Debug] All raw scores (before threshold):")
        score_page_pairs = [(float(score), page) for score, page in zip(scores, pages)]
        score_page_pairs.sort(key=lambda x: x[0], reverse=True)
        for i, (score, page) in enumerate(score_page_pairs, 1):
            print(f"  [{i}] Score: {score:.6f} | Title: {page.get('title', 'N/A')[:80]}")
        
        self.logger.log("-----Original-----")
        if self.logger._enable:
            for page in pages:
                self.logger.log(f'{page["score"]:.3f} + {page["title"]}')
        max_score = float("-inf")
        for score, search_result in zip(scores, pages):
            score = float(score)
            max_score = max(max_score, score)
            search_result["score"] = score
        if max_score == float("-inf"):
            raise RuntimeError("Reranker returned all invalid scores (all -inf)")
        threshold_score = max_score * relative_threshold
        
        # DEBUG: Log threshold info
        print(f"\n[Reranker Debug] Threshold calculation:")
        print(f"  Max score: {max_score:.6f}")
        print(f"  Relative threshold: {relative_threshold}")
        print(f"  Threshold score: {threshold_score:.6f}")
        
        results: list[SearchResult] = []
        for search_result in pages:
            if search_result["score"] >= threshold_score:
                results.append(search_result)
        results = sorted(results, key=lambda r: r["score"], reverse=True)
        
        # DEBUG: Log filtered results
        print(f"\n[Reranker Debug] Filtered results (after threshold, {len(results)}/{len(pages)}):")
        for i, page in enumerate(results, 1):
            print(f"  [{i}] Score: {page['score']:.6f} | Title: {page.get('title', 'N/A')[:80]}")
        
        self.logger.log("-----Reorder-----")
        if self.logger._enable:
            for page in results:
                self.logger.log(f'{page["score"]:.3f} + {page["title"]}')
        return results
    async def keywords(self, question: str, params: GenerationParams, threshold: float = 0.5) -> list[KeywordInfo]:
        num_queries = params.get("max_query", 1)
        copy_params = copy.deepcopy(params)
        copy_params.update(KEYWORDS_PARAMS) #type:ignore
        prompt = KEYWORD_TEMPLATE.format(question=question)
        text = ""
        async for chunk in await self(
            call_type=CallType.KEYWORDS, 
            instruction=KEYWORDS_INTRUCTION, 
            prompt=KEYWORDS_PREFIX.replace("{num}", str(num_queries))+prompt, 
            params=copy_params
        ):
            text += chunk
        try:
            self.logger.log(text)
            result: list[KeywordInfo] = json.loads(extract_json(text))
            for item in result:
                self.logger.log(item)
            return result
        except:
            print(text)
            traceback.print_exc()
            return []
    async def _heristic_rerank_page(self, pages: list[SearchResult], query: str, relative_threshold: float, params: GenerationParams) -> list[SearchResult]:
        """Rerank search results using embedding similarity"""
        self.logger.log("-----Original-----")
        if self.logger._enable:
            for page in pages:
                self.logger.log(f'{page["score"]:.3f} + {page["title"]}')
        import numpy as np
        import re
        def normalize_text(text: str) -> str:
            text = text.lower().strip()
            text = re.sub(r"[^a-zA-Z0-9\u00C0-\u1EF9\s\.,;]", " ", text)
            text = re.sub(r"\s+", " ", text).strip()
            return text
        def detect_school(query: str, schools: dict) -> str | None:
            """Detect school from query using predefined keywords"""
            for school, aliases in schools.items():
                if any(alias in query for alias in aliases):
                    return school
            return None
        schools = {
            school: [normalize_text(alias) for alias in aliases]
            for school, aliases in json.load(open(f"{BASE_PATH}files/school_alias.json", "r", encoding="utf-8")).items()
        }
        embedding = ws_pipeline.retriever.web_retriever.pipeline._rag.embedding
        
        # If no embedding model available, return original results
        if not embedding:
            return pages
        
        query_norm = normalize_text(query)
        detected_school = detect_school(query_norm, schools)
        
        try:
            query_emb = embedding.embed_query(query_norm)
            max_score = 0
            for page in pages:
                title = page.get("title", "") or ""
                desc = page.get("description", "") or ""
                url = page.get("url", "") or ""

                # Chuẩn hóa
                title_norm = normalize_text(title)
                desc_norm = normalize_text(desc)
                url_norm = normalize_text(url)

                # Semantic embedding
                title_emb = embedding.embed_query(title_norm) if title_norm else None
                desc_emb = embedding.embed_query(desc_norm) if desc_norm else None
                url_emb = embedding.embed_query(url_norm) if url_norm else None

                # Cosine similarity
                def cos_sim(a, b):
                    norm_a = np.linalg.norm(a)
                    norm_b = np.linalg.norm(b)
                    if norm_a == 0 or norm_b == 0:
                        return 0.0
                    return float(np.dot(a, b) / (norm_a * norm_b))

                score = 0.0
                weights = {"title": 0.5, "desc": 0.3, "url": 0.2}
                if title_emb is not None:
                    score += cos_sim(query_emb, title_emb) * weights["title"]
                if desc_emb is not None:
                    score += cos_sim(query_emb, desc_emb) * weights["desc"]
                if url_emb is not None:
                    score += cos_sim(query_emb, url_emb) * weights["url"]

                # Heuristic ưu tiên trường trong query
                if detected_school:
                    aliases = [normalize_text(a) for a in schools.get(detected_school, [])]
                    if any(a in text for a in aliases for text in [url_norm, title_norm, desc_norm]):
                        score += 0.5
                    else:
                        for school, other_aliases in schools.items():
                            if school != detected_school:
                                other_aliases_norm = [normalize_text(a) for a in other_aliases]
                                if any(a in text for a in other_aliases_norm for text in [url_norm, title_norm, desc_norm]):
                                    score -= 0.5

                # Heuristic boost
                if any(kw in query_norm for kw in ["tuyển sinh", "ngành đào tạo"]):
                    if "tuyensinh247" in url_norm:
                        score += 0.1
                    if url_norm.endswith(".edu") or ".edu.vn" in url_norm:
                        score += 0.2
                page["score"] = score
                max_score = max(score, max_score)
            threshold_score = max_score * relative_threshold
            # Sort theo score giảm dần
            results = []
            for page in pages:
                if page["score"] >= threshold_score:
                    results.append(page)
            results = sorted(results, key=lambda x: x["score"], reverse=True)
            self.logger.log("-----Reorder-----")
            if self.logger._enable:
                for page in results:
                    self.logger.log(f'{page["score"]:.3f} + {page["title"]}')
            return results
            
        except Exception:
            # If any error occurs, return original results
            return pages
    async def rerank_page(self, pages: list[SearchResult], query: str, relative_threshold: float, params: GenerationParams) -> list[SearchResult]:
        use_llm_rerank = params.get("llm_rerank", False)
        if use_llm_rerank:
            return await self._llm_rerank_page(pages, query, relative_threshold, params)
        else:
            return await self._heristic_rerank_page(pages, query, relative_threshold, params)

##### Pipeline

In [ ]:
import json as _json
import copy as _copy
import traceback as _traceback

class CombinedProtocol(ModelProtocol, KeywordModelProtocol, PageRerankModelProtocol, RouterModelProtocol):
    pass


# ──────────────────────────────────────────────────────────────────
# Multi-Hop adapters (gọi VLLMModel với prompt từ instruction.decomposer)
# ──────────────────────────────────────────────────────────────────
DECOMPOSER_PARAMS = {"temperature": 0.3, "top_p": 0.9, "max_tokens": 1024}
FACT_EXTRACTOR_PARAMS = {"temperature": 0.2, "top_p": 0.9, "max_tokens": 256}
REASONER_PARAMS = {"temperature": 0.1, "top_p": 0.9, "max_tokens": 1024}


class _DecomposerAdapter:
    """Adapter để MultiHopOrchestrator gọi LLM phân rã câu hỏi thành DAG sub-q."""
    def __init__(self, vllm_model):
        self._m = vllm_model

    async def decompose(self, question: str, params):
        copy_params = _copy.deepcopy(dict(params))
        copy_params.update(DECOMPOSER_PARAMS)
        prompt = DECOMPOSER_TEMPLATE.format(question=question)
        text = ""
        try:
            stream = await self._m(
                call_type="decomposer",
                instruction=DECOMPOSER_INSTRUCTION + "\n\n" + DECOMPOSER_PREFIX,
                prompt=prompt,
                params=copy_params,
            )
            async for chunk in stream:
                text += chunk
        except Exception:
            _traceback.print_exc()
            return None
        print(f"[Decomposer raw] {text[:400]}...")
        _debug_block("DECOMPOSER_RAW_OUTPUT", text, limit=_debug_int_env("BOT_DEBUG_PROMPT_CHARS", 20000))
        plan = parse_plan_json(text, max_sub=5)
        if plan is None:
            print("[Decomposer] parse failed → single-hop")
        return plan


class _FactExtractorAdapter:
    """Adapter trích fact ngắn từ top chunks cho hop tiếp theo."""
    def __init__(self, vllm_model):
        self._m = vllm_model

    async def extract(self, sub_q_text: str, rag_sources: list, params, evidence_type: str = "factual"):
        if not rag_sources:
            return "", 0.0
        top = rag_sources[:5]
        context = "\n\n".join(
            f"### Source {idx}: {c.get('title', '')} | {c.get('url', '')}\n"
            f"{_select_relevant_evidence_text(sub_q_text, c.get('text', '') or '', 1200)}"
            for idx, c in enumerate(top, 1)
        )
        prompt = FACT_EXTRACTOR_TEMPLATE.format(question=sub_q_text, context=context)
        _debug_block(
            "FACT_EXTRACTOR_INPUT",
            _debug_json({
                "question": sub_q_text,
                "evidence_type": evidence_type,
                "source_count": len(rag_sources),
                "selected_count": len(top),
                "selected_sources": [
                    {
                        "title": c.get("title", ""),
                        "url": c.get("url", ""),
                        "chunk_index": c.get("chunk_index"),
                        "text_chars": len(c.get("text", "") or ""),
                    }
                    for c in top
                ],
            }),
            limit=0,
        )
        copy_params = _copy.deepcopy(dict(params))
        copy_params.update(FACT_EXTRACTOR_PARAMS)
        text = ""
        try:
            stream = await self._m(
                call_type="fact_extractor",
                instruction=FACT_EXTRACTOR_INSTRUCTION,
                prompt=prompt,
                params=copy_params,
            )
            async for chunk in stream:
                text += chunk
        except Exception:
            _traceback.print_exc()
            return "", 0.0
        _debug_block("FACT_EXTRACTOR_RAW_OUTPUT", text, limit=_debug_int_env("BOT_DEBUG_PROMPT_CHARS", 20000))
        try:
            obj = _json.loads(extract_json(text))
            ans = str(obj.get("answer", "")).strip()
            if isinstance(obj.get("items"), list):
                items_ans = " ; ".join(
                    f"{item.get('name', '')}: {item.get('value', '')}".strip(": ")
                    for item in obj.get("items", [])
                    if isinstance(item, dict) and (item.get("name") or item.get("value"))
                ).strip()
                if items_ans:
                    ans = items_ans
            conf = float(obj.get("confidence", 0.0))
            _debug_block("FACT_EXTRACTOR_PARSED", _debug_json({"answer": ans, "confidence": conf}), limit=0)
            return ans, conf
        except Exception:
            _debug_block("FACT_EXTRACTOR_PARSE_FAILED", text, limit=_debug_int_env("BOT_DEBUG_PROMPT_CHARS", 20000))
            return "", 0.0


class _ReasonerAdapter:
    """Adapter cho reasoning hop: loc/so sanh/tong hop tren evidence cua deps."""
    def __init__(self, vllm_model):
        self._m = vllm_model

    async def reason(self, original_question: str, sub_q_text: str, evidence_blocks: list, params):
        if not evidence_blocks:
            return "", 0.0
        joined = "\n\n".join(evidence_blocks)
        prompt = REASONER_TEMPLATE.format(
            original_question=original_question,
            sub_question=sub_q_text,
            evidence=joined,
        )
        _debug_block(
            "REASONER_INPUT",
            _debug_json({
                "original_question": original_question,
                "sub_question": sub_q_text,
                "evidence_blocks": len(evidence_blocks),
                "evidence_chars": len(joined),
            }),
            limit=0,
        )
        _debug_block("REASONER_EVIDENCE", joined, limit=_debug_int_env("BOT_DEBUG_PROMPT_CHARS", 20000))
        copy_params = _copy.deepcopy(dict(params))
        copy_params.update(REASONER_PARAMS)
        text = ""
        try:
            stream = await self._m(
                call_type="reasoner",
                instruction=REASONER_INSTRUCTION,
                prompt=prompt,
                params=copy_params,
            )
            async for chunk in stream:
                text += chunk
        except Exception:
            _traceback.print_exc()
            return "", 0.0
        _debug_block("REASONER_RAW_OUTPUT", text, limit=_debug_int_env("BOT_DEBUG_PROMPT_CHARS", 20000))
        try:
            obj = _json.loads(extract_json(text))
            ans = str(obj.get("answer", "")).strip()
            conf = float(obj.get("confidence", 0.0))
            _debug_block("REASONER_PARSED", _debug_json({"answer": ans, "confidence": conf}), limit=0)
            return ans, conf
        except Exception:
            _debug_block("REASONER_PARSE_FAILED", text, limit=_debug_int_env("BOT_DEBUG_PROMPT_CHARS", 20000))
            return "", 0.0


class CustomQA:
    def __init__(self, model_protocol: CombinedProtocol) -> None:
        self.logger = CmdLogger("QA")
        web_retriever = WebRetriever(model_protocol, model_protocol)
        # Initialize LocalRetriever with RAG config for semantic search
        local_retriever = LocalRetriever(rag_config=rag_config)
        self.retriever = RouterRetriever(
            model_protocol,
            web_retriever,
            local_retriever
        )
        self.llm_call = model_protocol
        # Sufficiency Gate: reuse cross-encoder của ChunkRanker để tiết kiệm VRAM
        try:
            self.sufficiency = SufficiencyGate(
                chunk_ranker=web_retriever.pipeline.chunk_ranker,
                config=SufficiencyConfig(),
            )
        except Exception:
            _traceback.print_exc()
            self.sufficiency = None
        # Multi-Hop Orchestrator (opt-in qua params["use_multi_hop"]=True)
        self.multi_hop = MultiHopOrchestrator(
            base_retriever=self.retriever,
            decomposer=_DecomposerAdapter(model_protocol),
            fact_extractor=_FactExtractorAdapter(model_protocol),
            reasoner=_ReasonerAdapter(model_protocol),
            config=MultiHopConfig(enabled=True, max_hops=3, max_sub_questions=5),
        )
        print(
            "[CustomQA] multi_hop=ON | sufficiency=",
            "ON" if self.sufficiency is not None else "OFF",
        )

    async def start(self):
        await self.retriever.web_retriever.start()

    async def inference(self, prompt: str, request: WorkerChatRequest) -> AsyncGenerator[str, None]:
        text = ""
        hard_answer = request["params"].get("_hard_no_answer_text")
        if hard_answer:
            yield str(hard_answer)
            return
        async for chunk in await self.llm_call(
            call_type=CallType.READER,
            instruction=READER_INSTRUCTION,
            prompt=prompt,
            params=request["params"]
        ):
            text += chunk
            yield chunk

    async def pre_inference(
        self,
        question: str,
        stream_id: str,
        params: GenerationParams
    ) -> tuple[str, ModelPreOutput]:
        _debug_block(
            "REQUEST_START",
            _debug_json({
                "stream_id": stream_id,
                "question": question,
                "params": dict(params),
            }),
            limit=0,
        )
        # Multi-hop (opt-in). Khi tắt hoặc decomposer fail → Orchestrator tự fallback single-hop.
        web_sources, rag_sources, multi_hop_trace = await self.multi_hop.retrieve(
            question, params
        )
        if multi_hop_trace is not None:
            print(
                f"[MULTI-HOP DONE] sub_q={len(multi_hop_trace.plan.sub_questions)} | "
                f"hops={multi_hop_trace.hop_count} | rag={len(rag_sources)} | web={len(web_sources)}"
            )
        if multi_hop_trace is not None:
            _debug_block(
                "MULTI_HOP_TRACE",
                _debug_json({
                    "hop_count": multi_hop_trace.hop_count,
                    "sub_questions": [
                        {
                            "id": sq.id,
                            "text": sq.text,
                            "depends_on": sq.depends_on,
                            "resolver": sq.resolver,
                            "evidence_type": sq.evidence_type,
                            "rewritten_text": (multi_hop_trace.per_sub_q.get(sq.id).rewritten_text if multi_hop_trace.per_sub_q.get(sq.id) else ""),
                            "fact": (multi_hop_trace.per_sub_q.get(sq.id).fact if multi_hop_trace.per_sub_q.get(sq.id) else ""),
                            "confidence": (multi_hop_trace.per_sub_q.get(sq.id).confidence if multi_hop_trace.per_sub_q.get(sq.id) else 0.0),
                            "web_count": (len(multi_hop_trace.per_sub_q.get(sq.id).web_sources) if multi_hop_trace.per_sub_q.get(sq.id) else 0),
                            "rag_count": (len(multi_hop_trace.per_sub_q.get(sq.id).rag_sources) if multi_hop_trace.per_sub_q.get(sq.id) else 0),
                        }
                        for sq in multi_hop_trace.plan.sub_questions
                    ],
                }),
                limit=0,
            )
        else:
            _debug_block("MULTI_HOP_TRACE", "trace=None (single-hop fallback or disabled)")
        if multi_hop_trace is not None:
            q_norm = _normalize_for_match(question)
            strict_multi_hop = any(t in q_norm for t in ["so sanh", "danh sach", "xep hang", "top", "loc"])
            needs_metric = any(t in q_norm for t in ["diem chuan", "diem trung tuyen", "hoc phi"])
            missing_reasoning = []
            missing_metric_subq = []
            for sq in multi_hop_trace.plan.sub_questions:
                res = multi_hop_trace.per_sub_q.get(sq.id)
                fact = (res.fact or "").strip() if res else ""
                fact_norm = _normalize_for_match(fact)
                if any(t in fact_norm for t in ["khong co thong tin", "khong tim thay", "chua tim thay", "khong duoc cung cap", "unknown", "none", "n/a"]):
                    fact = ""
                sq_norm = _normalize_for_match(sq.text)
                if sq.resolver == "reasoning" and not fact:
                    missing_reasoning.append(sq.id)
                elif any(t in sq_norm for t in ["diem chuan", "diem trung tuyen", "hoc phi"]) and not fact:
                    missing_metric_subq.append(sq.id)
            if strict_multi_hop and needs_metric and (missing_reasoning or missing_metric_subq):
                params["_hard_no_answer_text"] = "Tôi chưa tìm thấy đủ thông tin đáng tin cậy để trả lời đầy đủ câu hỏi này."
                print(f"[MULTIHOP_HARD_STOP] missing_reasoning={missing_reasoning} missing_metric_subq={missing_metric_subq}")
        rag_sources = _augment_rag_with_web_table_evidence(
            question,
            web_sources,
            rag_sources,
            max_sources=int(params.get("reader_table_rescue_max_sources", 2)),
            max_chars=int(params.get("reader_table_rescue_max_chars", 6000)),
        )
        _debug_source_list("FINAL_WEB_SOURCES", web_sources)
        _debug_source_list("FINAL_RAG_CHUNKS", rag_sources)
        # Sufficiency check: log only - vẫn forward sang reader để reader tự nói "không tìm thấy" nếu cần.
        if self.sufficiency is not None and rag_sources:
            try:
                suff = self.sufficiency.check(question, rag_sources)
                params["sufficiency_score"] = suff.score
                q_norm = _normalize_for_match(question)
                strict_need = any(t in q_norm for t in ["so sanh", "danh sach", "xep hang", "top", "loc", "diem chuan", "hoc phi"])
                strict_low_score = strict_need and suff.score < float(params.get("strict_sufficiency_min_score", 0.35))
                params["low_confidence"] = (not suff.sufficient) or strict_low_score
                print(
                    f"[Sufficiency] sufficient={suff.sufficient} score={suff.score:.3f} reason={suff.reason}"
                )
            except Exception:
                _traceback.print_exc()
        print("\n" + "=" * 80)
        print(f"[RAG CHUNKS] Total {len(rag_sources)} chunks selected for reader:")
        for idx, chunk in enumerate(rag_sources, 1):
            title = chunk.get("title", "N/A")
            url = chunk.get("url", "N/A")
            chunk_idx = chunk.get("chunk_index", "N/A")
            text_preview = chunk.get("text", "")[:400]
            print("-" * 80)
            print(f"[Chunk {idx}] title={title} | url={url} | chunk_index={chunk_idx}")
            print("Text preview:\n" + text_preview)
        if not rag_sources:
            print("[RAG CHUNKS] No chunks selected.")
        print("=" * 80)
        context = SourceFormat()(rag_sources)
        prompt = READER_TEMPLATE.format(context=context, question=question)
        if params.get("low_confidence"):
            prompt = (
                "LUU Y: bang chung truy xuat co the chua du. Neu khong du co so, hay tra loi ro 'Toi chua tim thay thong tin' thay vi suy doan.\n\n"
                + prompt
            )
        print("\n" + "=" * 80)
        print("[RAG CONTEXT] Formatted chunks sent to reader:")
        print("-" * 80)
        print(context if context.strip() else "[Empty context]")
        print("=" * 80)
        if _debug_full_prompt_enabled():
            print("[FINAL PROMPT] Qwen4B input:")
            print("-" * 80)
            print(prompt)
            print("=" * 80 + "\n")
            _debug_block("READER_SYSTEM_PROMPT", READER_INSTRUCTION, limit=_debug_int_env("BOT_DEBUG_PROMPT_CHARS", 20000))
            _debug_block("READER_CONTEXT", context, limit=_debug_int_env("BOT_DEBUG_PROMPT_CHARS", 20000))
            _debug_block("READER_FINAL_PROMPT", prompt, limit=_debug_int_env("BOT_DEBUG_PROMPT_CHARS", 20000))
        self.logger.start()
        pre_output: ModelPreOutput = {
            "generation_params": params,
            "web_sources": web_sources,
            "rag_sources": rag_sources,
            "extra_data": {
            },
            "result_url": stream_id,
        }
        return prompt, pre_output


### Final

In [ ]:
engine_args = AsyncEngineArgs(
    model=MODEL_ID,
    tensor_parallel_size=2,
    gpu_memory_utilization=0.6,  # Reduced from 0.7 to 0.6 to prevent OOM when processing multiple requests
    max_model_len=32768,
    max_num_seqs=4,  # Limit concurrent sequences to reduce memory usage
    enable_lora=True,
    max_lora_rank=32,
    max_loras=1
)
vllm_model = VLLMModel()
vllm_model.init(engine_args)

[Reranker] Searching for adapter, BASE_PATH=/kaggle/working/, cwd=/kaggle/working
[Reranker] Checking: app/package/lora/qwen_reranker_06b_v1 (exists: False)
[Reranker] Path does not exist: app/package/lora/qwen_reranker_06b_v1
[Reranker] Checking: /kaggle/working/app/package/lora/qwen_reranker_06b_v1 (exists: False)
[Reranker] Path does not exist: /kaggle/working/app/package/lora/qwen_reranker_06b_v1
[Reranker] Checking: /kaggle/working/lora/qwen_reranker_06b_v1 (exists: True)
[Reranker] ✓ Found adapter at: /kaggle/working/lora/qwen_reranker_06b_v1
[Reranker] Found adapter at: /kaggle/working/lora/qwen_reranker_06b_v1
WARNING 12-02 16:08:20 [config.py:3392] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 12-02 16:08:20 [config.py:3443] Casting torch.bfloat16 to torch.float16.
INFO 12-02 16:08:20 [config.py:1604] Using max model len 32768
INFO 12-02 16:08:21 [llm_engine.py:228] Initializing a V

2025-12-02 16:08:25.972720: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764691705.992962    1709 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764691705.998914    1709 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 12-02 16:08:31 [__init__.py:235] Automatically detected platform cuda.
(VllmWorkerProcess pid=1709) INFO 12-02 16:08:31 [multiproc_worker_utils.py:226] Worker ready; awaiting tasks
(VllmWorkerProcess pid=1709) INFO 12-02 16:08:33 [cuda.py:346] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
(VllmWorkerProcess pid=1709) INFO 12-02 16:08:33 [cuda.py:395] Using XFormers backend.
INFO 12-02 16:08:34 [__init__.py:1375] Found nccl from library libnccl.so.2
(VllmWorkerProcess pid=1709) INFO 12-02 16:08:34 [__init__.py:1375] Found nccl from library libnccl.so.2
(VllmWorkerProcess pid=1709) INFO 12-02 16:08:34 [pynccl.py:70] vLLM is using nccl==2.26.2
INFO 12-02 16:08:34 [pynccl.py:70] vLLM is using nccl==2.26.2
INFO 12-02 16:08:34 [custom_all_reduce_utils.py:246] reading GPU P2P access cache from /root/.cache/vllm/gpu_p2p_access_cache_for_0,1.json
(VllmWorkerProcess pid=1709) INFO 12-02 16:08:34 [custom_all_reduce_utils.py:246] reading GPU P2P access cache from /root/.cache

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 12-02 16:08:53 [default_loader.py:262] Loading weights took 17.67 seconds
(VllmWorkerProcess pid=1709) INFO 12-02 16:08:53 [default_loader.py:262] Loading weights took 17.83 seconds
INFO 12-02 16:08:53 [punica_selector.py:19] Using PunicaWrapperGPU.
(VllmWorkerProcess pid=1709) INFO 12-02 16:08:53 [punica_selector.py:19] Using PunicaWrapperGPU.
INFO 12-02 16:08:54 [model_runner.py:1115] Model loading took 3.8589 GiB and 18.195583 seconds
(VllmWorkerProcess pid=1709) INFO 12-02 16:08:54 [model_runner.py:1115] Model loading took 3.8589 GiB and 18.428353 seconds
(VllmWorkerProcess pid=1709) INFO 12-02 16:09:08 [worker.py:295] Memory profiling takes 13.81 seconds
(VllmWorkerProcess pid=1709) INFO 12-02 16:09:08 [worker.py:295] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.70) = 10.32GiB
(VllmWorkerProcess pid=1709) INFO 12-02 16:09:08 [worker.py:295] model weights take 3.86GiB; non_torch_memory takes 0.11GiB; PyTorch activation peak memory t

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

(VllmWorkerProcess pid=1709) INFO 12-02 16:10:07 [model_runner.py:1537] Graph capturing finished in 55 secs, took 0.40 GiB
INFO 12-02 16:10:07 [model_runner.py:1537] Graph capturing finished in 55 secs, took 0.40 GiB
INFO 12-02 16:10:07 [llm_engine.py:424] init engine (profile, create kv cache, warmup model) took 73.27 seconds


Server wrapper

In [ ]:
ws_pipeline = CustomQA(vllm_model)
await ws_pipeline.start()
import uuid
class ServerModelImplement(ServerModel):  
    def __init__(self) -> None:
        self.request_storage: dict[str, tuple[str, WorkerChatRequest, ModelPreOutput]] = {}
    async def pre_inference(self, request: WorkerChatRequest) -> ModelPreOutput:
        stream_id = str(uuid.uuid4())
        params = request["params"]
        params.setdefault("enable_quality_gate", False)
        params.setdefault("quality_log", params["enable_quality_gate"])
        params.setdefault("quality_min_score", 0.58)
        params.setdefault("quality_min_relevance", 0.25)
        params.setdefault("quality_min_trust", 0.50)
        params.setdefault("quality_strict_mode", True)
        params.setdefault("quality_max_docs", params.get("k_pages", 3))
        params.setdefault("source_safety_filter", True)
        params.setdefault("quality_force_for_rich_media", True)
        params.setdefault("hybrid_retrieval", True)
        params.setdefault("auto_multi_hop", True)
        params.setdefault("multi_hop_complexity_threshold", 2)
        params.setdefault("reader_table_rescue_max_sources", 2)
        params.setdefault("reader_table_rescue_max_chars", 6000)

        _debug_block(
            "SERVER_REQUEST",
            _debug_json({
                "stream_id": stream_id,
                "text": request.get("text"),
                "params": dict(params),
                "forward_kwargs": request.get("forward_kwargs"),
            }),
            limit=0,
        )
        print(params)
        print(
            f"[DocumentQualityGate] enable_quality_gate={params['enable_quality_gate']} | "
            f"quality_log={params['quality_log']} | "
            f"min_score={params['quality_min_score']} | "
            f"min_relevance={params['quality_min_relevance']} | "
            f"min_trust={params['quality_min_trust']} | "
            f"strict_mode={params['quality_strict_mode']} | "
            f"quality_max_docs={params['quality_max_docs']}"
        )
        try:
            prompt, pre_output = await ws_pipeline.pre_inference(
                request["text"],
                stream_id,
                request["params"]
            ) 
            self.request_storage[stream_id] = (prompt, request, pre_output)
            return pre_output
        finally:
            # Cleanup memory after pre_inference (reranker, embedding operations)
            import gc
            import torch
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    async def inference(self, stream_id: str) -> AsyncGenerator[str, None]:
        prompt, request, pre_output = self.request_storage.pop(stream_id)
        generator = ws_pipeline.inference(prompt, request)
        total = ""
        try:
            async for chunk in generator:
                total += chunk
                yield chunk
        finally:
            # Store chat data when finish
            model_output: ModelOutput = {
                **pre_output,
                "text": total
            }
            data: WorkerStoreChatData = {
                "forward_kwargs": request["forward_kwargs"],
                "model_output": model_output
            }
            _debug_block(
                "FINAL_RESPONSE",
                _debug_json({
                    "stream_id": stream_id,
                    "response_chars": len(total),
                    "response": total,
                    "rag_sources": len(pre_output.get("rag_sources", [])),
                    "web_sources": len(pre_output.get("web_sources", [])),
                }),
                limit=0,
            )
            _debug_block(
                "STORE_CHAT_DATA_SUMMARY",
                _debug_json({
                    "stream_id": stream_id,
                    "forward_kwargs": request.get("forward_kwargs"),
                    "model_output_keys": list(model_output.keys()),
                }),
                limit=0,
            )
            await self.store(data)
            
            # Cleanup memory after each inference to prevent accumulation
            import gc
            import torch
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()  # Ensure all CUDA operations are complete

[ChunkRanker] Loaded model: BAAI/bge-reranker-v2-m3
[ImageOCR] Enabled with VISION_AGENT_API_KEY


Connect to server

In [ ]:
server_model = ServerModelImplement()
app = construct_app(
    server_domain=DOMAIN,
    info=CLIENT_INFO,
    server_model=server_model,
    init_tasks=[],
    shutdown_tasks=[],
    is_local=IS_LOCAL
)
# CORS policy
from fastapi.middleware.cors import CORSMiddleware
origins = [
    "http://127.0.0.1:8000", # I don't know why, but this won't work if not add this while in kaggle
    DOMAIN
]
app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)
import uvicorn
import asyncio
import sys
import time

# ── Kaggle: bật/tắt server (không cần Stop Execution) ─────────────────────
# "start"   → chạy uvicorn (mặc định nền, cell xong ngay)
# "stop"    → ngắt server, giữ kernel + model đã load
# "restart" → stop rồi start lại
# "skip"    → không đụng server (chạy cell khác)
KAGGLE_SERVER_ACTION = "start"  # start | stop | restart | skip
KAGGLE_SERVER_BLOCK = True        # True = giu cell dang chay de log hien truc tiep nhu flow test cu
KAGGLE_LOG_TO_FILE = True
KAGGLE_LOG_APPEND = False          # False = moi lan run cell tao log test moi
KAGGLE_LOG_PATH = f"{BASE_PATH}vllm4_runtime.txt"

_kaggle_uvicorn_server = globals().get("_kaggle_uvicorn_server")
_kaggle_server_task = globals().get("_kaggle_server_task")
_kaggle_log_file = globals().get("_kaggle_log_file")

class _NotebookTee:
    def __init__(self, primary, file_obj):
        self.primary = primary
        self.file_obj = file_obj
        self._is_notebook_tee = True

    def write(self, data):
        self.primary.write(data)
        self.file_obj.write(data)
        self.file_obj.flush()

    def flush(self):
        self.primary.flush()
        self.file_obj.flush()

    def isatty(self):
        return self.primary.isatty() if hasattr(self.primary, "isatty") else False

    def fileno(self):
        return self.primary.fileno()

    @property
    def encoding(self):
        return getattr(self.primary, "encoding", "utf-8")

    @property
    def errors(self):
        return getattr(self.primary, "errors", "replace")

    def __getattr__(self, name):
        return getattr(self.primary, name)

def install_notebook_log_tee(path: str) -> None:
    global _kaggle_log_file
    if not KAGGLE_LOG_TO_FILE:
        return
    if _kaggle_log_file is not None and not _kaggle_log_file.closed:
        _kaggle_log_file.close()
    stdout_primary = getattr(sys.stdout, "primary", sys.stdout)
    stderr_primary = getattr(sys.stderr, "primary", sys.stderr)
    mode = "a" if KAGGLE_LOG_APPEND else "w"
    _kaggle_log_file = open(path, mode, encoding="utf-8")
    _kaggle_log_file.write("\n" + "=" * 80 + "\n")
    _kaggle_log_file.write(f"[LOG_SESSION_START] {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    _kaggle_log_file.flush()
    sys.stdout = _NotebookTee(stdout_primary, _kaggle_log_file)
    sys.stderr = _NotebookTee(stderr_primary, _kaggle_log_file)
    print(f"[Log] Tee enabled -> {path}")

install_notebook_log_tee(KAGGLE_LOG_PATH)

async def stop_kaggle_server(timeout: float = 20.0) -> None:
    global _kaggle_uvicorn_server, _kaggle_server_task
    if _kaggle_uvicorn_server is not None:
        _kaggle_uvicorn_server.should_exit = True
    if _kaggle_server_task is not None and not _kaggle_server_task.done():
        try:
            await asyncio.wait_for(_kaggle_server_task, timeout=timeout)
        except asyncio.TimeoutError:
            _kaggle_server_task.cancel()
            print(f"[Server] Stop timeout ({timeout}s) — đã cancel task")
    _kaggle_uvicorn_server = None
    _kaggle_server_task = None
    print("[Server] Đã ngắt (kernel vẫn chạy, có thể chạy cell test)")

async def start_kaggle_server(*, block: bool = False) -> None:
    global _kaggle_uvicorn_server, _kaggle_server_task
    if _kaggle_server_task is not None and not _kaggle_server_task.done():
        print("[Server] Đang chạy rồi — dùng KAGGLE_SERVER_ACTION='stop' hoặc 'restart'")
        return
    uvicorn_config = uvicorn.Config(app, port=NGROK_PORT)
    _kaggle_uvicorn_server = uvicorn.Server(uvicorn_config)
    _kaggle_server_task = asyncio.create_task(_kaggle_uvicorn_server.serve())
    print(f"[Server] Uvicorn port={NGROK_PORT} | block={block}")
    if block:
        await _kaggle_server_task

print("[Server] Connect to server")
print(
    "[Server] DocumentQualityGate log pipeline: "
    "keyword extraction -> web search -> document scoring -> rag filter"
)
print(
    "[Server] Default DocumentQualityGate params: "
    "enable_quality_gate=True, quality_min_score=0.58, "
    "quality_min_relevance=0.25, quality_min_trust=0.50, quality_strict_mode=True"
)

# ──────────────────────────────────────────────────────────────────
# MULTI-HOP RETRIEVAL ENGINE — visual trace hook
# Bọc orchestrator để in tiến trình từng bước (DECOMPOSE → EXECUTE → AGGREGATE → FORWARD)
# ngay trong notebook log mỗi khi request bật `params["use_multi_hop"] = True`.
# ──────────────────────────────────────────────────────────────────
def _install_multi_hop_visual_trace(pipeline):
    orch = getattr(pipeline, "multi_hop", None)
    if orch is None:
        print("[MultiHop] orchestrator not found on pipeline → skip visual trace")
        return
    _orig_retrieve = orch.retrieve
    _orig_execute_subq = orch._execute_subq

    def _short(s, n=160):
        s = (s or "").replace("\n", " ").strip()
        return s if len(s) <= n else s[: n - 1] + "…"

    async def _execute_subq_logged(sq, resolved, params):
        deps = ",".join(f"#{d}" for d in sq.depends_on) if sq.depends_on else "—"
        print(f"  ▶ SQ#{sq.id} START [{sq.resolver}/{sq.evidence_type}] deps={deps}")
        print(f"      Q-raw: {_short(sq.text)}")
        res = await _orig_execute_subq(sq, resolved, params)
        if res.rewritten_text and res.rewritten_text != sq.text:
            print(f"      Q-rewritten: {_short(res.rewritten_text)}")
        print(
            f"  ✓ SQ#{sq.id} DONE  web={len(res.web_sources)} "
            f"rag={len(res.rag_sources)} conf={res.confidence:.2f}"
        )
        if res.fact:
            print(f"      Fact[{sq.id}]: {_short(res.fact, 200)}")
        else:
            print(f"      Fact[{sq.id}]: (none)")
        _debug_source_list(f"SQ{sq.id}_WEB_SOURCES", res.web_sources)
        _debug_source_list(f"SQ{sq.id}_RAG_CHUNKS", res.rag_sources)
        return res

    async def _retrieve_logged(question, params):
        enabled = params.get("use_multi_hop", orch.config.enabled)
        if not enabled:
            print("[MultiHop] DISABLED for this request → single-hop pipeline")
            return await _orig_retrieve(question, params)

        auto_gate = bool(params.get("auto_multi_hop", getattr(orch.config, "auto_detect_complexity", True)))
        force_multi_hop = bool(params.get("force_multi_hop", False))
        if auto_gate and not force_multi_hop:
            from data_retriever.multi_hop import _multi_hop_complexity_score
            score, reasons = _multi_hop_complexity_score(question)
            threshold = int(params.get("multi_hop_complexity_threshold", getattr(orch.config, "complexity_threshold", 2)))
            if score < threshold:
                print(f"[MultiHop] AutoGate SIMPLE -> single-hop (score={score}, threshold={threshold}, reasons={reasons})")
                return await _orig_retrieve(question, params)
            print(f"[MultiHop] AutoGate COMPLEX -> multi-hop (score={score}, threshold={threshold}, reasons={reasons})")

        print("\n" + "═" * 80)
        print("║ [MULTI-HOP] PIPELINE START")
        print(f"║ Question: {_short(question, 200)}")
        print("║ Config: max_hops={mh}, max_sub_q={ms}, subq_max_query={sq}, "
              "fact_extract={fe}".format(
                  mh=orch.config.max_hops,
                  ms=orch.config.max_sub_questions,
                  sq=orch.config.subq_max_query,
                  fe=orch.config.enable_fact_extraction,
              ))
        print("═" * 80)
        print("[Step 1/4] DECOMPOSE → LLM phân rã câu hỏi thành DAG sub-questions")

        web, rag, trace = await _orig_retrieve(question, params)

        if trace is None:
            print("[MultiHop] Decomposer trivial/failed → fell back to single-hop\n")
            return web, rag, trace

        print("─" * 80)
        print(
            f"[DECOMPOSE result] {len(trace.plan.sub_questions)} sub-question(s), "
            f"sẽ chạy trong {trace.hop_count} hop(s)"
        )
        print("DAG plan:")
        for sq in trace.plan.sub_questions:
            deps = "→".join(f"#{d}" for d in sq.depends_on) if sq.depends_on else "(root)"
            print(f"  • SQ#{sq.id} [{sq.resolver}/{sq.evidence_type}] deps={deps}")
            print(f"      {_short(sq.text)}")
        print("─" * 80)
        print("[Step 2/4] EXECUTE → chạy hops theo topological order (sub-q ready song song)")
        print("            (chi tiết từng SQ được in real-time trong _execute_subq_logged)")

        print("─" * 80)
        print(
            f"[Step 3/4] AGGREGATE → dedup theo (url, chunk_index): "
            f"web={len(web)} chunks, rag={len(rag)} chunks"
        )
        for sq in trace.plan.sub_questions:
            r = trace.per_sub_q.get(sq.id)
            if r is None:
                continue
            print(
                f"  ⤷ SQ#{sq.id}: contributed web={len(r.web_sources)} "
                f"rag={len(r.rag_sources)} fact={'yes' if r.fact else 'no'}"
            )
        print("─" * 80)
        print("[Step 4/4] FORWARD → SufficiencyGate kiểm tra → Reader Qwen3-4B sinh câu trả lời")
        print("═" * 80 + " [MULTI-HOP] DONE\n")
        return web, rag, trace

    orch.retrieve = _retrieve_logged
    orch._execute_subq = _execute_subq_logged
    print("[MultiHop] visual trace hook installed ✓")

_install_multi_hop_visual_trace(ws_pipeline)

print("\n" + "═" * 80)
print("║  MULTI-HOP RETRIEVAL ENGINE — pipeline overview")
print("║  ────────────────────────────────────────────────────────────────────")
print("║  [1] DECOMPOSE  → LLM phân rã câu hỏi phức tạp thành DAG sub-questions")
print("║                   resolver ∈ {local_db | web | hybrid | reasoning}")
print("║                   parse JSON an toàn + kiểm tra cycle (DFS) trước khi chạy")
print("║  [2] EXECUTE    → chạy theo topological order (sub-q ready chạy song song)")
print("║                   • local_db / web / hybrid → retrieve evidence (FAISS+BM25+RRF)")
print("║                   • reasoning              → tổng hợp facts từ deps (synthetic chunk)")
print("║                   • fact_extractor (LLM)   → trích fact ngắn cho bridge-entity")
print("║                   • rewrite                → chèn fact vào sub-q phụ thuộc")
print("║                                              (tiếng Việt tự nhiên: '... (tức X)')")
print("║                   • fallback               → local_db rỗng tự thử lại web")
print("║  [3] AGGREGATE  → dedup theo (url, chunk_index) → web + rag list cuối")
print("║  [4] FORWARD    → SufficiencyGate (cross-encoder) → Reader Qwen3-4B")
print("║")
print("║  Cấu hình mặc định: max_hops=3, max_sub_questions=5, subq_max_query=1")
print("║  Opt-in: bật bằng params['use_multi_hop'] = True trong request body.")
print("║  Khi disabled / decomposer trivial / decomposer fail → tự fallback single-hop.")
print("═" * 80 + "\n")

_action = str(KAGGLE_SERVER_ACTION).strip().lower()
print(f"[Server] KAGGLE_SERVER_ACTION={_action!r} | KAGGLE_SERVER_BLOCK={KAGGLE_SERVER_BLOCK}")
if _action == "skip":
    print("[Server] skip — không start/stop server")
elif _action == "stop":
    await stop_kaggle_server()
elif _action == "restart":
    await stop_kaggle_server()
    await start_kaggle_server(block=KAGGLE_SERVER_BLOCK)
elif _action == "start":
    await start_kaggle_server(block=KAGGLE_SERVER_BLOCK)
else:
    raise ValueError(
        f"KAGGLE_SERVER_ACTION không hợp lệ: {_action!r} "
        "(dùng start | stop | restart | skip)"
    )
print("[Server] Cell xong — đổi KAGGLE_SERVER_ACTION rồi Run lại cell này để stop/start")

INFO:     Started server process [1638]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8002 (Press CTRL+C to quit)


Domain: https://cymosely-interventional-richie.ngrok-free.dev
{'model_id': 'Qwen/Qwen3-4B', 'use_websearch': True, 'use_localdb': False, 'max_query': 3, 'query_score_threshold': 0.5, 'engine_type': 'google', 'domain_restrict': False, 'school_domain': False, 'llm_rerank': True, 'page_score_threshold': 0.5, 'chunk_score_threshold': 0.5, 'k_docs': 5, 'k_pages': 3, 'page_rerank': False, 'chunk_rerank': True, 'include_pdf': True, 'include_image': True, 'merge_table': True, 'merge_neighbor': True, 'max_tokens': 2048, 'temperature': 0.5, 'top_p': 0.9, 'top_k': 40, 'max_history': 8}
[VLLM] CallType.KEYWORDS | Instruction length: 168 | Prompt length: 8627 | kwargs: None
INFO 12-02 16:11:35 [chat_utils.py:473] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
INFO 12-02 16:11:35 [async_llm_engine.py:209] Added request 187372baf4174a628c9028fb21b21790.
INFO 12-02 16:11:36 [metrics.py:386] Avg prompt throughput: 33.2 tokens/s, 

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-Reranker-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[Reranker] Loading from /kaggle/working/lora/qwen_reranker_06b_v1
[Reranker] Loading adapter state dict to check shapes...
[Reranker] Detected num_labels from base_model.model.score.weight shape: 1
[Reranker] Resizing score layer: 2 -> 1
[Reranker] Score layer resized successfully to 1 labels
[Reranker] Detected num_labels from base_model.model.score.weight shape: 1
[Reranker] Found modules_to_save: base_model.model.score.weight -> score.weight (shape: torch.Size([1, 1024]))
[Reranker] Loading 1 modules_to_save weights into model...
[Reranker] Missing keys: ['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.self_attn.q_norm.weight', 'model.layers.0.self_attn.k_norm.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.input_layernorm.weight'

/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['target_parameters'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


[Reranker] PEFT config loaded: ['default']
[Reranker] Keys with 'score' in model: ['base_model.model.score.original_module.weight', 'base_model.model.score.modules_to_save.default.weight']
[Reranker] Merging LoRA weights into base model...
[Reranker] ✓ score.weight present after merge (shape: torch.Size([1, 1024]))
[Reranker] score.weight norm: 0.647461 (should be > 0 if loaded correctly)
[Reranker] ✓ score.weight preserved after merge, no need to re-load
[Reranker] Successfully loaded and merged LoRA adapter from /kaggle/working/lora/qwen_reranker_06b_v1

[Reranker Debug] Query: điểm chuẩn uet 2022
[Reranker Debug] Number of pages: 10
[Reranker Debug] Sample page texts (first 3):

  Page 1:
    Title: ĐIỂM TRÚNG TUYỂN VÀO ĐẠI HỌC CHÍNH QUY NĂM 2022 THEO ...
    Description: Trường Đại học Công nghệ, ĐHQGHN (VNU-UET) ... Điểm chuẩn trúng tuyển được công bố theo thang điểm 3
    URL: https://tuyensinh.uet.vnu.edu.vn/phuong-thuc-xet-tuyen/diem-trung-tuyen-vao-dai-hoc-chinh-quy-nam-2022-t

###### Note 
If the instruction is too long -> freeze. Seem like vLLM cache instruction, but we still don't know why it only freeze after some requests. (Seem like cache problem, use llm serve would cause it)
